# School Uniform Detector Training - Custom YOLOv8-Style Model

This notebook prepares the school-uniform dataset, trains a manually implemented anchor-free detector from random initialization, validates it on a clean validation split, and builds a Windows 11 deployment package. Every model layer, loss, update rule, decoder, suppression step, and metric is coded with low-level tensor arithmetic. The workflow uses only `train` and `val`; no test split is generated.

## 1. Environment and Custom Configuration

In [1]:
# Cell 1 - Environment and fixed configuration
from __future__ import annotations

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ.setdefault("PYTHONUTF8", "1")

IMAGE_DATASET_DIR = Path("/content/drive/MyDrive/DATN2/dataset_images")
LABELS_ALL_DIR = Path("/content/drive/MyDrive/DATN2/labels/all")
OUTPUT_ROOT = Path("/content/drive/MyDrive/DATN2/yolov8_uniform_training_output")
WORK_DIR = Path("/content/uniform_yolo_dataset")
DATA_YAML_PATH = WORK_DIR / "data.yaml"
FRAMEWORK_LOG_DIR = OUTPUT_ROOT / "framework_logs"
ZIP_PATH = Path("/content/drive/MyDrive/DATN2/yolov8_uniform_windows_package.zip")

CLASS_NAMES = [
    "ao_so_mi_trang",
    "ao_doan_thanh_nien",
    "quan_tay_dai_den",
    "khan_quang_do",
    "quan_short_tay_den",
    "quan_dai_trang",
]
NUM_CLASSES = len(CLASS_NAMES)

BAD_IDS = {357, 4105}
ALLOWED_LEGACY_IDS = {1193, 1194, 2329}
TRAIN_ONLY_ID_RANGE = range(4904, 5301)
RANDOM_SEED = 42
VALIDATION_RATIO = 0.10

RUN_NAME = "uniform_detector_training"
VALIDATION_RUN_NAME = "uniform_detector_validation"
IMG_SIZE = 640
EPOCHS = 200
BATCH_SIZE = 8
WORKERS = 2
PATIENCE = 40
CACHE = False
PREVIEW_SAMPLES_PER_SPLIT = 8
SAMPLE_PREDICTION_COUNT = 8
MODEL_CONFIG = {
    "name": "uniform_custom_c2f_panfpn_anchor_free",
    "channels": [16, 32, 64, 96, 128],
    "repeats": [1, 1, 2, 2],
    "class_count": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "reg_max": 12,
    "strides": [8, 16, 32],
    "assignment_top_k": 5,
}

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FRAMEWORK_LOG_DIR.mkdir(parents=True, exist_ok=True)

package_specs = {
    "numpy": "numpy",
    "pandas": "pandas",
    "yaml": "pyyaml",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "cv2": "opencv-python-headless",
}
missing_packages = [package for module, package in package_specs.items() if importlib.util.find_spec(module) is None]
install_log = FRAMEWORK_LOG_DIR / "package_install.log"
if missing_packages:
    process = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        text=True,
        capture_output=True,
    )
    install_log.write_text("STDOUT\n" + process.stdout + "\nSTDERR\n" + process.stderr, encoding="utf-8")
    if process.returncode != 0:
        raise RuntimeError(f"Package installation failed. See {install_log}")
else:
    install_log.write_text("All required packages were already available.\n", encoding="utf-8")

import torch

print("Environment prepared for custom random-initialized training.")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Mounted at /content/drive
Environment prepared.
CUDA available: True
GPU: Tesla T4


## 2. Dataset, Model, Training, and Evaluation Utilities

In [2]:
# Cell 2 - Shared utilities for scanning, splitting, reporting, logging, and packaging
from __future__ import annotations

import contextlib
import hashlib
import io
import json
import logging
import math
import random
import re
import shutil
import traceback
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
import yaml

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ID_PATTERN = re.compile(r"id(\d+)(?:_|$)")
INTEGER_PATTERN = re.compile(r"^[+-]?\d+$")
BOUNDARY_TOLERANCE = 1e-6


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, range):
        return [value.start, value.stop - 1]
    if isinstance(value, set):
        return sorted(value)
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=json_default), encoding="utf-8")


def parse_image_id_from_stem(stem: str) -> int | None:
    match = ID_PATTERN.search(stem)
    return int(match.group(1)) if match else None


def stable_score(record: dict[str, Any], seed: int) -> int:
    key = f"{seed}:{record['image_stem']}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:8], "big")


def list_source_images(image_dir: Path) -> list[Path]:
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Image source directory not found: {image_dir}")
    return sorted(path for path in image_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)


def detect_duplicate_image_stems(image_paths: list[Path]) -> dict[str, list[str]]:
    by_stem: dict[str, list[str]] = defaultdict(list)
    for image_path in image_paths:
        by_stem[image_path.stem.casefold()].append(str(image_path))
    return {stem: paths for stem, paths in by_stem.items() if len(paths) > 1}


def inspect_reference_classes(reference_candidates: list[Path], expected_classes: list[str]) -> dict[str, Any]:
    report = {
        "reference_file_found": False,
        "reference_file_path": "",
        "reference_classes": [],
        "required_class_order_present": False,
        "extra_reference_classes_ignored_for_training": [],
    }
    for candidate in reference_candidates:
        if candidate.is_file():
            observed = [line.strip() for line in candidate.read_text(encoding="utf-8-sig", errors="replace").splitlines() if line.strip()]
            report.update(
                {
                    "reference_file_found": True,
                    "reference_file_path": str(candidate),
                    "reference_classes": observed,
                    "required_class_order_present": observed[: len(expected_classes)] == expected_classes,
                    "extra_reference_classes_ignored_for_training": observed[len(expected_classes):],
                }
            )
            if not report["required_class_order_present"]:
                raise RuntimeError(f"Reference classes do not begin with the required six-class order: {candidate}")
            return report
    return report


def validate_yolo_label_file(label_path: Path, class_names: list[str]) -> dict[str, Any]:
    try:
        raw_text = label_path.read_text(encoding="utf-8-sig", errors="replace")
    except OSError as exc:
        return {"is_empty": False, "is_valid": False, "boxes": [], "errors": [f"could not read label file: {exc}"]}

    non_empty_lines = [(idx, line.strip()) for idx, line in enumerate(raw_text.splitlines(), start=1) if line.strip()]
    if not non_empty_lines:
        return {"is_empty": True, "is_valid": True, "boxes": [], "errors": []}

    errors: list[str] = []
    boxes: list[dict[str, Any]] = []
    for line_number, line in non_empty_lines:
        row_errors: list[str] = []
        parts = line.split()
        if len(parts) != 5:
            row_errors.append(f"line {line_number}: expected exactly 5 values, got {len(parts)}")
        if len(parts) == 5:
            class_token, *coord_tokens = parts
            class_id = None
            if not INTEGER_PATTERN.fullmatch(class_token):
                row_errors.append(f"line {line_number}: class_id is not an integer: {class_token!r}")
            else:
                class_id = int(class_token)
                if class_id not in range(len(class_names)):
                    row_errors.append(f"line {line_number}: class_id {class_id} outside 0-{len(class_names) - 1}")
            coords = None
            try:
                coords = tuple(float(value) for value in coord_tokens)
            except ValueError:
                row_errors.append(f"line {line_number}: coordinates are not numeric")
            if coords is not None:
                x_center, y_center, width, height = coords
                if not all(math.isfinite(value) for value in coords):
                    row_errors.append(f"line {line_number}: coordinates must be finite")
                if not (0.0 <= x_center <= 1.0 and 0.0 <= y_center <= 1.0):
                    row_errors.append(f"line {line_number}: x_center and y_center must be in [0, 1]")
                if not (0.0 < width <= 1.0 and 0.0 < height <= 1.0):
                    row_errors.append(f"line {line_number}: width and height must be > 0 and <= 1")
                left = x_center - width / 2.0
                right = x_center + width / 2.0
                top = y_center - height / 2.0
                bottom = y_center + height / 2.0
                if left < -BOUNDARY_TOLERANCE or top < -BOUNDARY_TOLERANCE or right > 1.0 + BOUNDARY_TOLERANCE or bottom > 1.0 + BOUNDARY_TOLERANCE:
                    row_errors.append(f"line {line_number}: bounding box extends outside normalized image boundaries")
            if not row_errors and coords is not None and class_id is not None:
                x_center, y_center, width, height = coords
                boxes.append({"class_id": int(class_id), "class_name": class_names[int(class_id)], "x_center": x_center, "y_center": y_center, "width": width, "height": height})
        errors.extend(row_errors)
    return {"is_empty": False, "is_valid": len(errors) == 0, "boxes": boxes, "errors": errors}


def count_classes(boxes: list[dict[str, Any]], class_names: list[str]) -> dict[str, int]:
    counts = {class_name: 0 for class_name in class_names}
    for box in boxes:
        counts[class_names[int(box["class_id"])]] += 1
    return counts


def make_record(image_path: Path, label_path: Path, parsed_id: int | None, status: str, split_eligibility: str, validation_errors: list[str] | None, object_count: int, class_ids_present: list[int] | None, class_counts: dict[str, int], split_eligibility_reasons: list[str] | None = None) -> dict[str, Any]:
    return {
        "source_image_path": str(image_path),
        "expected_label_path": str(label_path),
        "image_file_name": image_path.name,
        "image_stem": image_path.stem,
        "image_extension": image_path.suffix.lower(),
        "parsed_id": parsed_id,
        "original_record_status": status,
        "split_eligibility": split_eligibility,
        "split_eligibility_reasons": split_eligibility_reasons or [split_eligibility],
        "assigned_split": "",
        "validation_errors": validation_errors or [],
        "object_count": int(object_count),
        "class_ids_present": sorted(class_ids_present or []),
        "class_counts": class_counts,
    }


def scan_dataset(image_dir: Path, labels_all_dir: Path, output_root: Path, class_names: list[str], bad_ids: set[int], train_only_id_range: range) -> dict[str, Any]:
    output_root.mkdir(parents=True, exist_ok=True)
    if not labels_all_dir.is_dir():
        raise FileNotFoundError(f"Annotation source directory not found: {labels_all_dir}")
    image_paths = list_source_images(image_dir)
    duplicate_stems = detect_duplicate_image_stems(image_paths)
    if duplicate_stems:
        conflict_path = output_root / "duplicate_image_stem_conflicts.json"
        write_json(conflict_path, duplicate_stems)
        raise RuntimeError(f"Duplicate image stems make same-stem label matching ambiguous. See {conflict_path}")
    label_files = sorted(path for path in labels_all_dir.glob("*.txt") if path.is_file())
    records: list[dict[str, Any]] = []
    status_counts = Counter()
    matching_label_count = 0
    for image_path in image_paths:
        parsed_id = parse_image_id_from_stem(image_path.stem)
        label_path = labels_all_dir / f"{image_path.stem}.txt"
        if label_path.exists():
            matching_label_count += 1
        zero_counts = {class_name: 0 for class_name in class_names}
        if parsed_id in bad_ids:
            status_counts["excluded_bad_id"] += 1
            records.append(make_record(image_path, label_path, parsed_id, "excluded_bad_id", "excluded_bad_id", ["known annotation error ID excluded"], 0, [], zero_counts))
            continue
        if not label_path.exists():
            status_counts["missing_label"] += 1
            records.append(make_record(image_path, label_path, parsed_id, "missing_label", "excluded_missing_label", ["matching same-stem label file not found"], 0, [], zero_counts))
            continue
        validation = validate_yolo_label_file(label_path, class_names)
        boxes = validation["boxes"]
        class_ids_present = sorted({int(box["class_id"]) for box in boxes})
        class_counts = count_classes(boxes, class_names)
        if validation["is_empty"]:
            status_counts["empty_negative_label"] += 1
            reasons = ["train_only_empty_negative"]
            if parsed_id in train_only_id_range:
                reasons.append("excluded_train_only_range")
            records.append(make_record(image_path, label_path, parsed_id, "empty_negative_label", "train_only_empty_negative", [], 0, [], zero_counts, reasons))
            continue
        if not validation["is_valid"]:
            status_counts["invalid_label"] += 1
            records.append(make_record(image_path, label_path, parsed_id, "invalid_label", "excluded_invalid_label", list(validation["errors"]), len(boxes), class_ids_present, class_counts))
            continue
        split_eligibility = "excluded_train_only_range" if parsed_id in train_only_id_range else "eligible_for_train_and_val"
        status_counts["valid_positive"] += 1
        records.append(make_record(image_path, label_path, parsed_id, "valid_positive", split_eligibility, [], len(boxes), class_ids_present, class_counts))
    summary = {
        "total_source_images_found": len(image_paths),
        "total_label_files_in_labels_all": len(label_files),
        "total_matching_label_files_found": matching_label_count,
        "record_status_counts": dict(status_counts),
        "supported_image_extensions": sorted(IMAGE_EXTENSIONS),
    }
    return {"records": records, "summary": summary}



def create_train_val_split(scan_result: dict[str, Any], class_names: list[str], bad_ids: set[int], allowed_legacy_ids: set[int], train_only_id_range: range, seed: int, validation_ratio: float) -> dict[str, Any]:
    records = scan_result["records"]
    expected_class_ids = set(range(len(class_names)))
    for record in records:
        record["assigned_split"] = ""
    usable_records = [record for record in records if record["original_record_status"] in {"valid_positive", "empty_negative_label"}]
    validation_candidates = [record for record in usable_records if record["original_record_status"] == "valid_positive" and record["parsed_id"] not in train_only_id_range]
    target_val_count = int(round(len(usable_records) * validation_ratio))
    if validation_candidates and target_val_count == 0 and len(usable_records) > 1:
        target_val_count = 1
    target_val_count = min(target_val_count, len(validation_candidates))
    selected_val: list[dict[str, Any]] = []
    selected_stems: set[str] = set()
    classes_with_candidates = sorted({class_id for record in validation_candidates for class_id in record["class_ids_present"]})
    unavailable_class_ids = sorted(expected_class_ids - set(classes_with_candidates))
    if target_val_count >= len(classes_with_candidates):
        for class_id in classes_with_candidates:
            choices = [record for record in validation_candidates if class_id in record["class_ids_present"] and record["image_stem"] not in selected_stems]
            if choices:
                chosen = min(choices, key=lambda record: stable_score(record, seed + class_id))
                selected_val.append(chosen)
                selected_stems.add(chosen["image_stem"])
    remaining = [record for record in validation_candidates if record["image_stem"] not in selected_stems]
    remaining.sort(key=lambda record: (stable_score(record, seed), record["image_stem"]))
    for record in remaining:
        if len(selected_val) >= target_val_count:
            break
        selected_val.append(record)
        selected_stems.add(record["image_stem"])
    val_stems = {record["image_stem"] for record in selected_val}
    train_records = [record for record in usable_records if record["image_stem"] not in val_stems]
    val_records = list(selected_val)
    for record in train_records:
        record["assigned_split"] = "train"
    for record in val_records:
        record["assigned_split"] = "val"
    train_stems = {record["image_stem"] for record in train_records}
    assert train_stems.isdisjoint(val_stems), "Train and validation splits overlap."
    assert not any(record["parsed_id"] in bad_ids for record in train_records + val_records), "BAD_IDS leaked into a usable split."
    assert not any(record["original_record_status"] == "missing_label" for record in train_records + val_records), "Missing-label record leaked into a split."
    assert not any(record["original_record_status"] == "invalid_label" for record in train_records + val_records), "Invalid-label record leaked into a split."
    assert not any(record["original_record_status"] == "empty_negative_label" for record in val_records), "Empty negative label leaked into validation."
    assert not any(record["parsed_id"] in train_only_id_range for record in val_records), "Train-only ID range leaked into validation."
    for legacy_id in sorted(allowed_legacy_ids):
        for record in [record for record in records if record["parsed_id"] == legacy_id]:
            assert record["original_record_status"] != "excluded_bad_id", f"Allowed legacy ID {legacy_id} was treated as a bad ID."
            assert record["split_eligibility"] != "excluded_bad_id", f"Allowed legacy ID {legacy_id} was excluded by ID filtering."
            if record["original_record_status"] in {"valid_positive", "empty_negative_label"}:
                assert record["assigned_split"] in {"train", "val"}, f"Allowed legacy ID {legacy_id} was not assigned despite being usable."
    val_classes = {class_id for record in val_records for class_id in record["class_ids_present"]}
    missing_candidate_class_ids = sorted(set(classes_with_candidates) - val_classes)
    if target_val_count >= len(classes_with_candidates):
        assert not missing_candidate_class_ids, f"Validation split is missing classes despite sufficient candidates: {missing_candidate_class_ids}"
    total_usable = len(train_records) + len(val_records)
    summary = {
        "target_validation_images": target_val_count,
        "train_images": len(train_records),
        "validation_images": len(val_records),
        "final_train_percentage": round((len(train_records) / total_usable) * 100, 4) if total_usable else 0.0,
        "final_validation_percentage": round((len(val_records) / total_usable) * 100, 4) if total_usable else 0.0,
        "classes_with_validation_candidates": classes_with_candidates,
        "validation_class_ids_present": sorted(val_classes),
        "validation_class_ids_missing": sorted(expected_class_ids - val_classes),
        "validation_class_ids_unavailable": unavailable_class_ids,
        "validation_class_names_unavailable": [class_names[class_id] for class_id in unavailable_class_ids],
        "validation_candidate_class_ids_missing_from_val": missing_candidate_class_ids,
    }
    return {"train": train_records, "val": val_records, "summary": summary}

def ensure_safe_work_dir(path: Path, allow_non_content_work_dir: bool = False) -> None:
    resolved = path.resolve()
    if resolved in {Path("/"), Path("/content"), Path("/content/drive")}:
        raise RuntimeError(f"Refusing to clean unsafe work directory: {resolved}")
    if not allow_non_content_work_dir and not str(resolved).startswith("/content/"):
        raise RuntimeError(f"Temporary YOLO dataset must stay under /content: {resolved}")


def materialize_yolo_dataset(splits: dict[str, Any], work_dir: Path, class_names: list[str], data_yaml_path: Path, bad_ids: set[int], train_only_id_range: range, allow_non_content_work_dir: bool = False) -> list[dict[str, Any]]:
    ensure_safe_work_dir(work_dir, allow_non_content_work_dir=allow_non_content_work_dir)
    if work_dir.exists():
        shutil.rmtree(work_dir)
    for split_name in ["train", "val"]:
        (work_dir / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (work_dir / "labels" / split_name).mkdir(parents=True, exist_ok=True)
    copied_manifest: list[dict[str, Any]] = []
    for split_name in ["train", "val"]:
        txt_lines: list[str] = []
        for record in splits[split_name]:
            source_image = Path(record["source_image_path"])
            source_label = Path(record["expected_label_path"])
            if not source_image.exists():
                raise FileNotFoundError(f"Selected image is missing: {source_image}")
            if not source_label.exists():
                raise FileNotFoundError(f"Selected label is missing: {source_label}")
            target_image = work_dir / "images" / split_name / source_image.name
            target_label = work_dir / "labels" / split_name / source_label.name
            shutil.copy2(source_image, target_image)
            shutil.copy2(source_label, target_label)
            txt_lines.append(str(target_image))
            copied_manifest.append({"split": split_name, "parsed_id": record["parsed_id"], "image_stem": record["image_stem"], "source_image": str(source_image), "source_label": str(source_label), "target_image": str(target_image), "target_label": str(target_label), "original_record_status": record["original_record_status"], "object_count": record["object_count"]})
        (work_dir / f"{split_name}.txt").write_text("\n".join(txt_lines) + ("\n" if txt_lines else ""), encoding="utf-8")
    (work_dir / "classes.txt").write_text("\n".join(class_names) + "\n", encoding="utf-8")
    data_yaml = {"path": str(work_dir), "train": "images/train", "val": "images/val", "nc": len(class_names), "names": {index: class_name for index, class_name in enumerate(class_names)}}
    data_yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")
    assert not (work_dir / "images" / "test").exists(), "images/test must not exist."
    assert not (work_dir / "labels" / "test").exists(), "labels/test must not exist."
    assert not (work_dir / "test.txt").exists(), "test.txt must not exist."
    generated_yaml = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
    assert "test" not in generated_yaml, "Generated data.yaml must not contain a test key."
    assert not any(item["parsed_id"] in bad_ids for item in copied_manifest), "BAD_IDS were copied."
    assert not any(item["split"] == "val" and item["parsed_id"] in train_only_id_range for item in copied_manifest), "Train-only ID range copied into validation."
    train_stems = {item["image_stem"] for item in copied_manifest if item["split"] == "train"}
    val_stems = {item["image_stem"] for item in copied_manifest if item["split"] == "val"}
    assert train_stems.isdisjoint(val_stems), "Copied train and validation sets overlap."
    for item in copied_manifest:
        assert Path(item["target_image"]).exists(), f"Copied image missing: {item['target_image']}"
        assert Path(item["target_label"]).exists(), f"Copied label missing: {item['target_label']}"
        if item["original_record_status"] == "empty_negative_label":
            assert Path(item["target_label"]).read_text(encoding="utf-8-sig", errors="replace").strip() == "", "Copied empty label did not remain empty."
    return copied_manifest


def flatten_record_for_csv(record: dict[str, Any], class_names: list[str]) -> dict[str, Any]:
    flat = {"source_image_path": record["source_image_path"], "expected_label_path": record["expected_label_path"], "image_stem": record["image_stem"], "parsed_id": record["parsed_id"], "original_record_status": record["original_record_status"], "split_eligibility": record["split_eligibility"], "split_eligibility_reasons": json.dumps(record.get("split_eligibility_reasons", []), ensure_ascii=False), "assigned_split": record["assigned_split"], "validation_errors": json.dumps(record["validation_errors"], ensure_ascii=False), "object_count": record["object_count"], "class_ids_present": json.dumps(record["class_ids_present"], ensure_ascii=False)}
    for class_name in class_names:
        flat[f"objects_{class_name}"] = int(record["class_counts"].get(class_name, 0))
    return flat


def object_counts_by_class(records: list[dict[str, Any]], class_names: list[str]) -> dict[str, int]:
    counts = {class_name: 0 for class_name in class_names}
    for record in records:
        for class_name in class_names:
            counts[class_name] += int(record["class_counts"].get(class_name, 0))
    return counts


def export_dataset_reports(scan_result: dict[str, Any], splits: dict[str, Any], copied_manifest: list[dict[str, Any]], output_root: Path, class_names: list[str], config: dict[str, Any], reference_class_report: dict[str, Any] | None = None) -> dict[str, Path]:
    output_root.mkdir(parents=True, exist_ok=True)
    records = scan_result["records"]
    train_records = splits["train"]
    val_records = splits["val"]
    usable_total = len(train_records) + len(val_records)
    status_counter = Counter(record["original_record_status"] for record in records)
    train_class_counts = object_counts_by_class(train_records, class_names)
    val_class_counts = object_counts_by_class(val_records, class_names)
    dataset_overview = {
        "total_source_images_found": scan_result["summary"]["total_source_images_found"],
        "total_matching_label_files_found": scan_result["summary"]["total_matching_label_files_found"],
        "total_label_files_in_labels_all": scan_result["summary"]["total_label_files_in_labels_all"],
        "valid_positive_images": int(status_counter.get("valid_positive", 0)),
        "empty_negative_label_images": int(status_counter.get("empty_negative_label", 0)),
        "missing_label_images": int(status_counter.get("missing_label", 0)),
        "invalid_label_images": int(status_counter.get("invalid_label", 0)),
        "excluded_bad_id_images": int(status_counter.get("excluded_bad_id", 0)),
        "train_images": len(train_records),
        "validation_images": len(val_records),
        "train_positive_images": sum(1 for record in train_records if record["original_record_status"] == "valid_positive"),
        "train_empty_negative_images": sum(1 for record in train_records if record["original_record_status"] == "empty_negative_label"),
        "validation_positive_images": sum(1 for record in val_records if record["original_record_status"] == "valid_positive"),
        "object_instances_per_class_train": train_class_counts,
        "object_instances_per_class_val": val_class_counts,
        "final_train_percentage": round((len(train_records) / usable_total) * 100, 4) if usable_total else 0.0,
        "final_validation_percentage": round((len(val_records) / usable_total) * 100, 4) if usable_total else 0.0,
        "random_seed": config["random_seed"],
        "bad_ids": sorted(config["bad_ids"]),
        "allowed_legacy_ids": sorted(config["allowed_legacy_ids"]),
        "legacy_ids_are_eligible_for_normal_split_processing": True,
        "train_only_id_range": [config["train_only_id_range"].start, config["train_only_id_range"].stop - 1],
        "model_format": config["model_format"],
        "no_test_split_created": True,
        "nc": len(class_names),
        "class_names": {index: class_name for index, class_name in enumerate(class_names)},
        "validation_class_ids_present": splits["summary"].get("validation_class_ids_present", []),
        "validation_class_ids_missing": splits["summary"].get("validation_class_ids_missing", []),
        "validation_class_ids_unavailable": splits["summary"].get("validation_class_ids_unavailable", []),
        "validation_class_names_unavailable": splits["summary"].get("validation_class_names_unavailable", []),
        "validation_candidate_class_ids_missing_from_val": splits["summary"].get("validation_candidate_class_ids_missing_from_val", []),
        "reference_classes_check": reference_class_report or {},
    }
    paths = {
        "dataset_validation_report_json": output_root / "dataset_validation_report.json",
        "dataset_validation_report_csv": output_root / "dataset_validation_report.csv",
        "split_summary_csv": output_root / "split_summary.csv",
        "split_manifest_csv": output_root / "split_manifest.csv",
        "class_distribution_csv": output_root / "class_distribution.csv",
        "class_distribution_png": output_root / "class_distribution.png",
        "dataset_overview_json": output_root / "dataset_overview.json",
        "training_configuration_json": output_root / "training_configuration.json",
    }
    write_json(paths["dataset_validation_report_json"], {"summary": dataset_overview, "records": records})
    pd.DataFrame([flatten_record_for_csv(record, class_names) for record in records]).to_csv(paths["dataset_validation_report_csv"], index=False, encoding="utf-8-sig")
    split_summary_rows = []
    for split_name, split_records in [("train", train_records), ("val", val_records)]:
        split_summary_rows.append({"split": split_name, "images": len(split_records), "labels": len(split_records), "positive_images": sum(1 for record in split_records if record["original_record_status"] == "valid_positive"), "empty_negative_images": sum(1 for record in split_records if record["original_record_status"] == "empty_negative_label"), "object_instances": sum(record["object_count"] for record in split_records), "percentage_of_usable_dataset": round((len(split_records) / usable_total) * 100, 4) if usable_total else 0.0})
    pd.DataFrame(split_summary_rows).to_csv(paths["split_summary_csv"], index=False, encoding="utf-8-sig")
    split_manifest_rows = []
    for split_name, split_records in [("train", train_records), ("val", val_records)]:
        for record in split_records:
            split_manifest_rows.append({"split": split_name, "image_stem": record["image_stem"], "parsed_id": record["parsed_id"], "source_image_path": record["source_image_path"], "expected_label_path": record["expected_label_path"], "original_record_status": record["original_record_status"], "split_eligibility": record["split_eligibility"], "object_count": record["object_count"], "class_ids_present": json.dumps(record["class_ids_present"], ensure_ascii=False)})
    pd.DataFrame(split_manifest_rows).to_csv(paths["split_manifest_csv"], index=False, encoding="utf-8-sig")
    class_distribution_rows = []
    for split_name, split_records in [("train", train_records), ("val", val_records)]:
        counts = object_counts_by_class(split_records, class_names)
        for class_id, class_name in enumerate(class_names):
            class_distribution_rows.append({"split": split_name, "class_id": class_id, "class_name": class_name, "instances": counts[class_name]})
    class_distribution_df = pd.DataFrame(class_distribution_rows)
    class_distribution_df.to_csv(paths["class_distribution_csv"], index=False, encoding="utf-8-sig")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    x_positions = list(range(len(class_names)))
    width = 0.36
    ax.bar([x - width / 2 for x in x_positions], [train_class_counts[c] for c in class_names], width=width, label="train", color="#2563eb")
    ax.bar([x + width / 2 for x in x_positions], [val_class_counts[c] for c in class_names], width=width, label="val", color="#dc2626")
    ax.set_xticks(x_positions)
    ax.set_xticklabels(class_names, rotation=18, ha="right")
    ax.set_ylabel("Object instances")
    ax.set_title("Class distribution by split")
    ax.legend()
    fig.tight_layout()
    fig.savefig(paths["class_distribution_png"], dpi=180)
    plt.close(fig)
    write_json(paths["dataset_overview_json"], dataset_overview)
    training_configuration = {"image_dataset_dir": str(config["image_dataset_dir"]), "labels_all_dir": str(config["labels_all_dir"]), "output_root": str(config["output_root"]), "temporary_work_dir": str(config["work_dir"]), "data_yaml_path": str(config["data_yaml_path"]), "model_format": config["model_format"], "training_method": "random initialization with custom tensor operations", "external_model_source": False, "prior_detector_weights_used": False, "architecture": config["architecture"], "nc": len(class_names), "class_names": class_names, "bad_ids": sorted(config["bad_ids"]), "allowed_legacy_ids": sorted(config["allowed_legacy_ids"]), "train_only_id_range": [config["train_only_id_range"].start, config["train_only_id_range"].stop - 1], "random_seed": config["random_seed"], "validation_ratio_target": config["validation_ratio"], "imgsz": config["img_size"], "epochs": config["epochs"], "batch": config["batch_size"], "workers": config["workers"], "patience": config["patience"], "cache": config["cache"], "augmentation": {"horizontal_flip_probability": 0.5, "cosine_learning_rate": True}, "splits": {"train": len(train_records), "val": len(val_records)}}
    write_json(paths["training_configuration_json"], training_configuration)
    assert set(pd.read_csv(paths["split_summary_csv"])["split"]) == {"train", "val"}, "split_summary.csv contains a non train/val split."
    assert set(pd.read_csv(paths["class_distribution_csv"])["split"]) == {"train", "val"}, "class_distribution.csv contains a non train/val split."
    assert set(pd.read_csv(paths["split_manifest_csv"])["split"]).issubset({"train", "val"}), "split_manifest.csv contains a non train/val split."
    return paths



@contextlib.contextmanager
def quiet_execution(log_dir: Path, log_name: str):
    log_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = log_dir / f"{timestamp}_{log_name}.log"
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    logger_buffer = io.StringIO()
    handler = logging.StreamHandler(logger_buffer)
    handler.setLevel(logging.DEBUG)
    loggers = [logging.getLogger()]
    old_levels = [logger.level for logger in loggers]
    for logger in loggers:
        logger.addHandler(handler)
    exception_text = ""
    try:
        with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
            yield log_path
    except Exception:
        exception_text = traceback.format_exc()
        raise
    finally:
        for logger, old_level in zip(loggers, old_levels):
            logger.removeHandler(handler)
            logger.setLevel(old_level)
        content = [f"log_name: {log_name}", f"\ncreated_at: {datetime.now().isoformat(timespec='seconds')}", "\n\nSTDOUT\n", stdout_buffer.getvalue(), "\nSTDERR\n", stderr_buffer.getvalue(), "\nLOGGER\n", logger_buffer.getvalue()]
        if exception_text:
            content.extend(["\nEXCEPTION\n", exception_text])
        log_path.write_text("".join(content), encoding="utf-8")


def export_training_curves(results_csv_path: Path, output_path: Path) -> Path:
    if not results_csv_path.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv_path}")
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = [column.strip() for column in results_df.columns]
    if results_df.empty:
        raise RuntimeError(f"results.csv is empty: {results_csv_path}")
    import matplotlib.pyplot as plt
    x_values = results_df["epoch"] if "epoch" in results_df.columns else results_df.index
    train_loss_cols = [column for column in results_df.columns if column.startswith("train/") and column.endswith("_loss")]
    val_loss_cols = [column for column in results_df.columns if column.startswith("val/") and column.endswith("_loss")]
    metric_cols = [column for column in results_df.columns if "mAP" in column or "precision" in column or "recall" in column]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if train_loss_cols or val_loss_cols:
        for column in train_loss_cols + val_loss_cols:
            axes[0].plot(x_values, results_df[column], label=column)
        axes[0].set_title("Training and validation losses")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].legend(fontsize=8)
    else:
        axes[0].text(0.5, 0.5, "No loss columns found", ha="center", va="center")
        axes[0].set_axis_off()
    if metric_cols:
        for column in metric_cols:
            axes[1].plot(x_values, results_df[column], label=column)
        axes[1].set_title("Validation metrics")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Metric")
        axes[1].legend(fontsize=8)
    else:
        axes[1].text(0.5, 0.5, "No metric columns found", ha="center", va="center")
        axes[1].set_axis_off()
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180)
    plt.close(fig)
    return output_path


def summarize_results_csv(results_csv_path: Path) -> dict[str, Any]:
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = [column.strip() for column in results_df.columns]
    if results_df.empty:
        return {"rows": 0}
    last_row = results_df.iloc[-1].to_dict()
    summary = {"rows": len(results_df)}
    for key in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        if key in last_row:
            summary[key] = float(last_row[key])
    return summary


def metric_value(metrics: dict[str, Any], candidates: list[str]) -> float | None:
    for candidate in candidates:
        if candidate in metrics and metrics[candidate] is not None:
            try:
                return float(metrics[candidate])
            except (TypeError, ValueError):
                return None
    return None


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Custom tensor detector runtime
import copy
import csv
import math
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch


CUSTOM_CHECKPOINT_FORMAT = "custom_anchor_free_detector_v1"
CUSTOM_FORMAT_VERSION = 1


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def manual_sigmoid(x):
    """Numerically stable sigmoid written directly from exp and division."""
    z = x.clamp(-60.0, 60.0)
    return 1.0 / (1.0 + torch.exp(-z))


def manual_silu(x):
    return x * manual_sigmoid(x)


def manual_softmax(x, dim=-1):
    shifted = x - x.amax(dim=dim, keepdim=True)
    numerator = torch.exp(shifted)
    return numerator / numerator.sum(dim=dim, keepdim=True).clamp_min(1e-12)


def manual_log_softmax(x, dim=-1):
    shifted = x - x.amax(dim=dim, keepdim=True)
    return shifted - torch.log(torch.exp(shifted).sum(dim=dim, keepdim=True).clamp_min(1e-12))


def stable_binary_loss(logits, targets):
    """Elementwise binary log loss using its stable max/log1p identity."""
    return logits.clamp_min(0.0) - logits * targets + torch.log1p(torch.exp(-logits.abs()))


def _component_children(value):
    if isinstance(value, TensorComponent):
        yield value
    elif isinstance(value, (list, tuple)):
        for item in value:
            yield from _component_children(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _component_children(item)


class TensorComponent:
    """Small recursive tensor container; all forward mathematics lives below."""

    def __init__(self):
        self.training = True

    def named_parameters(self, prefix=""):
        for name, value in self.__dict__.items():
            full = f"{prefix}.{name}" if prefix else name
            if isinstance(value, torch.Tensor) and value.requires_grad:
                yield full, value
            elif isinstance(value, TensorComponent):
                yield from value.named_parameters(full)
            elif isinstance(value, (list, tuple)):
                for index, item in enumerate(value):
                    if isinstance(item, TensorComponent):
                        yield from item.named_parameters(f"{full}.{index}")

    def named_tensors(self, prefix=""):
        for name, value in self.__dict__.items():
            full = f"{prefix}.{name}" if prefix else name
            if isinstance(value, torch.Tensor):
                yield full, value
            elif isinstance(value, TensorComponent):
                yield from value.named_tensors(full)
            elif isinstance(value, (list, tuple)):
                for index, item in enumerate(value):
                    if isinstance(item, TensorComponent):
                        yield from item.named_tensors(f"{full}.{index}")

    def parameters(self):
        return [value for _, value in self.named_parameters()]

    def state_dict(self):
        return {name: value.detach().cpu().clone() for name, value in self.named_tensors()}

    def load_state_dict(self, state):
        current = dict(self.named_tensors())
        missing = sorted(set(current) - set(state))
        unexpected = sorted(set(state) - set(current))
        if missing or unexpected:
            raise ValueError(f"State mismatch; missing={missing[:5]}, unexpected={unexpected[:5]}")
        with torch.no_grad():
            for name, value in current.items():
                incoming = state[name].to(device=value.device, dtype=value.dtype)
                if incoming.shape != value.shape:
                    raise ValueError(f"Shape mismatch for {name}: {incoming.shape} != {value.shape}")
                value.copy_(incoming)
        return self

    def to(self, device):
        for name, value in list(self.__dict__.items()):
            if isinstance(value, torch.Tensor):
                needs_grad = value.requires_grad
                moved = value.detach().to(device).requires_grad_(needs_grad)
                setattr(self, name, moved)
            elif isinstance(value, TensorComponent):
                value.to(device)
            elif isinstance(value, list):
                for item in value:
                    if isinstance(item, TensorComponent):
                        item.to(device)
        return self

    def train(self, mode=True):
        self.training = bool(mode)
        for value in self.__dict__.values():
            for child in _component_children(value):
                child.train(mode)
        return self

    def eval(self):
        return self.train(False)

    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)


def he_tensor(shape, fan_in):
    """Explicit He/Kaiming uniform initialization from random tensor arithmetic."""
    bound = math.sqrt(6.0 / max(1, fan_in))
    return ((torch.rand(*shape) * 2.0 - 1.0) * bound).requires_grad_(True)


def manual_pad_2d(x, padding):
    if padding == 0:
        return x
    batch, channels, height, width = x.shape
    side = torch.zeros(batch, channels, height, padding, device=x.device, dtype=x.dtype)
    x = torch.cat((side, x, side), dim=3)
    top = torch.zeros(batch, channels, padding, width + 2 * padding, device=x.device, dtype=x.dtype)
    return torch.cat((top, x, top), dim=2)


class ManualConv(TensorComponent):
    """Convolution explicitly formed as image patches multiplied by learned kernels."""

    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=None, bias=True):
        super().__init__()
        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.kernel_size = int(kernel_size)
        self.stride = int(stride)
        self.padding = self.kernel_size // 2 if padding is None else int(padding)
        fan_in = self.in_channels * self.kernel_size * self.kernel_size
        self.weight = he_tensor(
            (self.out_channels, self.in_channels, self.kernel_size, self.kernel_size), fan_in
        )
        self.bias = torch.zeros(self.out_channels, requires_grad=True) if bias else None

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise ValueError(f"Expected NCHW with {self.in_channels} channels, got {tuple(x.shape)}")
        padded = manual_pad_2d(x, self.padding)
        patches = padded.unfold(2, self.kernel_size, self.stride).unfold(
            3, self.kernel_size, self.stride
        )
        batch, _, out_h, out_w, _, _ = patches.shape
        rows = patches.permute(0, 2, 3, 1, 4, 5).reshape(batch * out_h * out_w, -1)
        kernels = self.weight.reshape(self.out_channels, -1)
        result = rows @ kernels.transpose(0, 1)
        if self.bias is not None:
            result = result + self.bias.reshape(1, -1)
        return result.reshape(batch, out_h, out_w, self.out_channels).permute(0, 3, 1, 2)


class ManualBatchNorm(TensorComponent):
    def __init__(self, channels, momentum=0.03, epsilon=1e-3):
        super().__init__()
        self.channels = int(channels)
        self.momentum = float(momentum)
        self.epsilon = float(epsilon)
        self.scale = torch.ones(channels, requires_grad=True)
        self.shift = torch.zeros(channels, requires_grad=True)
        self.running_mean = torch.zeros(channels)
        self.running_variance = torch.ones(channels)

    def forward(self, x):
        if self.training:
            mean = x.mean(dim=(0, 2, 3))
            centered = x - mean.reshape(1, -1, 1, 1)
            variance = (centered * centered).mean(dim=(0, 2, 3))
            with torch.no_grad():
                self.running_mean.mul_(1.0 - self.momentum).add_(self.momentum * mean.detach())
                self.running_variance.mul_(1.0 - self.momentum).add_(
                    self.momentum * variance.detach()
                )
        else:
            mean = self.running_mean
            variance = self.running_variance
            centered = x - mean.reshape(1, -1, 1, 1)
        normalized = centered / torch.sqrt(variance.reshape(1, -1, 1, 1) + self.epsilon)
        return normalized * self.scale.reshape(1, -1, 1, 1) + self.shift.reshape(1, -1, 1, 1)


class ConvNormAct(TensorComponent):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super().__init__()
        self.conv = ManualConv(in_channels, out_channels, kernel_size, stride, bias=False)
        self.norm = ManualBatchNorm(out_channels)

    def forward(self, x):
        return manual_silu(self.norm(self.conv(x)))


class Bottleneck(TensorComponent):
    def __init__(self, channels, shortcut=True):
        super().__init__()
        self.first = ConvNormAct(channels, channels, 3, 1)
        self.second = ConvNormAct(channels, channels, 3, 1)
        self.shortcut = bool(shortcut)

    def forward(self, x):
        y = self.second(self.first(x))
        return x + y if self.shortcut else y


class C2f(TensorComponent):
    def __init__(self, in_channels, out_channels, repeats=1):
        super().__init__()
        self.hidden = max(1, out_channels // 2)
        self.entry = ConvNormAct(in_channels, self.hidden * 2, 1, 1)
        self.blocks = [Bottleneck(self.hidden, True) for _ in range(int(repeats))]
        self.exit = ConvNormAct(self.hidden * (2 + len(self.blocks)), out_channels, 1, 1)

    def forward(self, x):
        split = self.entry(x)
        pieces = [split[:, : self.hidden], split[:, self.hidden :]]
        for block in self.blocks:
            pieces.append(block(pieces[-1]))
        return self.exit(torch.cat(pieces, dim=1))


def manual_max_pool_same(x, kernel_size=5):
    padding = kernel_size // 2
    batch, channels, height, width = x.shape
    side = torch.full(
        (batch, channels, height, padding), -float("inf"), device=x.device, dtype=x.dtype
    )
    padded = torch.cat((side, x, side), dim=3)
    top = torch.full(
        (batch, channels, padding, width + 2 * padding),
        -float("inf"),
        device=x.device,
        dtype=x.dtype,
    )
    padded = torch.cat((top, padded, top), dim=2)
    windows = padded.unfold(2, kernel_size, 1).unfold(3, kernel_size, 1)
    return windows.amax(dim=(-1, -2))


class SPPF(TensorComponent):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        hidden = max(1, in_channels // 2)
        self.entry = ConvNormAct(in_channels, hidden, 1, 1)
        self.exit = ConvNormAct(hidden * 4, out_channels, 1, 1)

    def forward(self, x):
        first = self.entry(x)
        second = manual_max_pool_same(first, 5)
        third = manual_max_pool_same(second, 5)
        fourth = manual_max_pool_same(third, 5)
        return self.exit(torch.cat((first, second, third, fourth), dim=1))


def manual_nearest_upsample(x, scale=2):
    return x.repeat_interleave(scale, dim=2).repeat_interleave(scale, dim=3)


class DetectionHead(TensorComponent):
    def __init__(self, in_channels, hidden, class_count, reg_max):
        super().__init__()
        self.box_stem = ConvNormAct(in_channels, hidden, 3, 1)
        self.class_stem = ConvNormAct(in_channels, hidden, 3, 1)
        self.box_output = ManualConv(hidden, 4 * reg_max, 1, 1, 0, True)
        self.class_output = ManualConv(hidden, class_count, 1, 1, 0, True)
        with torch.no_grad():
            self.class_output.bias.fill_(-4.0)

    def forward(self, x):
        return torch.cat((self.box_output(self.box_stem(x)), self.class_output(self.class_stem(x))), dim=1)


class CustomDetector(TensorComponent):
    """Compact C2f/SPPF backbone with a PAN/FPN neck and P3/P4/P5 heads."""

    def __init__(self, architecture):
        super().__init__()
        self.architecture = copy.deepcopy(architecture)
        channels = list(architecture["channels"])
        repeats = list(architecture["repeats"])
        class_count = int(architecture["class_count"])
        reg_max = int(architecture["reg_max"])
        c1, c2, c3, c4, c5 = channels
        self.stem = ConvNormAct(3, c1, 3, 2)
        self.down2 = ConvNormAct(c1, c2, 3, 2)
        self.stage2 = C2f(c2, c2, repeats[0])
        self.down3 = ConvNormAct(c2, c3, 3, 2)
        self.stage3 = C2f(c3, c3, repeats[1])
        self.down4 = ConvNormAct(c3, c4, 3, 2)
        self.stage4 = C2f(c4, c4, repeats[2])
        self.down5 = ConvNormAct(c4, c5, 3, 2)
        self.stage5 = C2f(c5, c5, repeats[3])
        self.sppf = SPPF(c5, c5)
        self.reduce5 = ConvNormAct(c5, c4, 1, 1)
        self.fuse4 = C2f(c4 + c4, c4, 1)
        self.reduce4 = ConvNormAct(c4, c3, 1, 1)
        self.fuse3 = C2f(c3 + c3, c3, 1)
        self.neck_down4 = ConvNormAct(c3, c3, 3, 2)
        self.pan4 = C2f(c3 + c4, c4, 1)
        self.neck_down5 = ConvNormAct(c4, c4, 3, 2)
        self.pan5 = C2f(c4 + c5, c5, 1)
        self.head3 = DetectionHead(c3, c3, class_count, reg_max)
        self.head4 = DetectionHead(c4, c4, class_count, reg_max)
        self.head5 = DetectionHead(c5, c5, class_count, reg_max)

    def forward(self, x):
        x1 = self.stem(x)
        x2 = self.stage2(self.down2(x1))
        p3 = self.stage3(self.down3(x2))
        p4 = self.stage4(self.down4(p3))
        p5 = self.sppf(self.stage5(self.down5(p4)))
        n4 = self.fuse4(torch.cat((manual_nearest_upsample(self.reduce5(p5)), p4), dim=1))
        n3 = self.fuse3(torch.cat((manual_nearest_upsample(self.reduce4(n4)), p3), dim=1))
        o4 = self.pan4(torch.cat((self.neck_down4(n3), n4), dim=1))
        o5 = self.pan5(torch.cat((self.neck_down5(o4), p5), dim=1))
        return [self.head3(n3), self.head4(o4), self.head5(o5)]


def flatten_detector_outputs(outputs, architecture):
    reg_max = int(architecture["reg_max"])
    class_count = int(architecture["class_count"])
    box_parts, class_parts, anchor_parts, stride_parts, level_parts = [], [], [], [], []
    for level, (output, stride) in enumerate(zip(outputs, architecture["strides"])):
        batch, _, height, width = output.shape
        flat = output.permute(0, 2, 3, 1).reshape(batch, height * width, 4 * reg_max + class_count)
        box_parts.append(flat[:, :, : 4 * reg_max])
        class_parts.append(flat[:, :, 4 * reg_max :])
        yy, xx = torch.meshgrid(
            torch.arange(height, device=output.device, dtype=output.dtype),
            torch.arange(width, device=output.device, dtype=output.dtype),
            indexing="ij",
        )
        anchor_parts.append(torch.stack(((xx + 0.5) * stride, (yy + 0.5) * stride), dim=-1).reshape(-1, 2))
        stride_parts.append(torch.full((height * width,), float(stride), device=output.device, dtype=output.dtype))
        level_parts.append(torch.full((height * width,), level, device=output.device, dtype=torch.long))
    return {
        "box_logits": torch.cat(box_parts, dim=1),
        "class_logits": torch.cat(class_parts, dim=1),
        "anchors": torch.cat(anchor_parts, dim=0),
        "strides": torch.cat(stride_parts, dim=0),
        "levels": torch.cat(level_parts, dim=0),
    }


def decode_flat_boxes(flat, architecture):
    reg_max = int(architecture["reg_max"])
    logits = flat["box_logits"].reshape(flat["box_logits"].shape[0], -1, 4, reg_max)
    probabilities = manual_softmax(logits, -1)
    bins = torch.arange(reg_max, device=logits.device, dtype=logits.dtype)
    distances = (probabilities * bins.reshape(1, 1, 1, -1)).sum(dim=-1)
    distances = distances * flat["strides"].reshape(1, -1, 1)
    anchors = flat["anchors"].reshape(1, -1, 2)
    return torch.stack(
        (
            anchors[:, :, 0] - distances[:, :, 0],
            anchors[:, :, 1] - distances[:, :, 1],
            anchors[:, :, 0] + distances[:, :, 2],
            anchors[:, :, 1] + distances[:, :, 3],
        ),
        dim=-1,
    )


def pairwise_iou(boxes_a, boxes_b):
    if boxes_a.numel() == 0 or boxes_b.numel() == 0:
        return torch.zeros((boxes_a.shape[0], boxes_b.shape[0]), device=boxes_a.device)
    left_top = torch.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])
    right_bottom = torch.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])
    intersection = (right_bottom - left_top).clamp_min(0.0).prod(dim=-1)
    area_a = (boxes_a[:, 2:] - boxes_a[:, :2]).clamp_min(0.0).prod(dim=-1)
    area_b = (boxes_b[:, 2:] - boxes_b[:, :2]).clamp_min(0.0).prod(dim=-1)
    return intersection / (area_a[:, None] + area_b[None, :] - intersection).clamp_min(1e-9)


def aligned_ciou(predicted, target):
    intersection_lt = torch.maximum(predicted[:, :2], target[:, :2])
    intersection_rb = torch.minimum(predicted[:, 2:], target[:, 2:])
    intersection = (intersection_rb - intersection_lt).clamp_min(0.0).prod(dim=-1)
    pred_wh = (predicted[:, 2:] - predicted[:, :2]).clamp_min(1e-6)
    target_wh = (target[:, 2:] - target[:, :2]).clamp_min(1e-6)
    union = pred_wh.prod(dim=-1) + target_wh.prod(dim=-1) - intersection
    iou = intersection / union.clamp_min(1e-9)
    pred_center = (predicted[:, :2] + predicted[:, 2:]) * 0.5
    target_center = (target[:, :2] + target[:, 2:]) * 0.5
    center_distance = ((pred_center - target_center) ** 2).sum(dim=-1)
    enclosure_lt = torch.minimum(predicted[:, :2], target[:, :2])
    enclosure_rb = torch.maximum(predicted[:, 2:], target[:, 2:])
    diagonal = ((enclosure_rb - enclosure_lt) ** 2).sum(dim=-1).clamp_min(1e-9)
    aspect = (4.0 / (math.pi * math.pi)) * (
        torch.atan(target_wh[:, 0] / target_wh[:, 1]) - torch.atan(pred_wh[:, 0] / pred_wh[:, 1])
    ) ** 2
    with torch.no_grad():
        alpha = aspect / (1.0 - iou + aspect).clamp_min(1e-9)
    return iou - center_distance / diagonal - alpha * aspect


def assign_training_targets(flat, targets, architecture, top_k=5):
    """Assign nearby in-box grid centers, ranked by center and object/stride compatibility."""
    batch = len(targets)
    anchor_count = flat["anchors"].shape[0]
    class_count = int(architecture["class_count"])
    device = flat["anchors"].device
    target_classes = torch.zeros(batch, anchor_count, class_count, device=device)
    target_boxes = torch.zeros(batch, anchor_count, 4, device=device)
    positive = torch.zeros(batch, anchor_count, dtype=torch.bool, device=device)
    assignment_cost = torch.full((batch, anchor_count), float("inf"), device=device)
    anchors = flat["anchors"]
    strides = flat["strides"]
    for batch_index, item in enumerate(targets):
        boxes = item["boxes"].to(device)
        classes = item["classes"].to(device)
        for gt_index in range(boxes.shape[0]):
            box = boxes[gt_index]
            center = (box[:2] + box[2:]) * 0.5
            wh = (box[2:] - box[:2]).clamp_min(1.0)
            inside = (
                (anchors[:, 0] >= box[0])
                & (anchors[:, 0] <= box[2])
                & (anchors[:, 1] >= box[1])
                & (anchors[:, 1] <= box[3])
            )
            center_cost = torch.sqrt((((anchors - center) / strides[:, None]) ** 2).sum(dim=1) + 1e-9)
            object_scale = torch.sqrt(wh[0] * wh[1])
            scale_cost = torch.abs(torch.log2((object_scale / (strides * 4.0)).clamp_min(1e-6)))
            cost = center_cost + 0.35 * scale_cost
            candidates = torch.nonzero(inside, as_tuple=False).reshape(-1)
            if candidates.numel() == 0:
                candidates = torch.argsort(center_cost)[:1]
            ordered = candidates[torch.argsort(cost[candidates])[: min(top_k, candidates.numel())]]
            for anchor_index in ordered.tolist():
                if cost[anchor_index] < assignment_cost[batch_index, anchor_index]:
                    old_class = torch.nonzero(target_classes[batch_index, anchor_index] > 0, as_tuple=False)
                    if old_class.numel():
                        target_classes[batch_index, anchor_index, old_class.reshape(-1)] = 0.0
                    class_index = int(classes[gt_index].item())
                    target_classes[batch_index, anchor_index, class_index] = 1.0
                    target_boxes[batch_index, anchor_index] = box
                    positive[batch_index, anchor_index] = True
                    assignment_cost[batch_index, anchor_index] = cost[anchor_index]
    return target_classes, target_boxes, positive


def detector_loss(outputs, targets, architecture):
    flat = flatten_detector_outputs(outputs, architecture)
    target_classes, target_boxes, positive = assign_training_targets(
        flat, targets, architecture, int(architecture.get("assignment_top_k", 5))
    )
    positive_count = positive.sum().clamp_min(1).to(flat["class_logits"].dtype)
    probabilities = manual_sigmoid(flat["class_logits"])
    focal_weight = (target_classes - probabilities).abs() ** 2
    alpha_weight = torch.where(target_classes > 0.0, 0.75, 0.25)
    classification = (stable_binary_loss(flat["class_logits"], target_classes) * focal_weight * alpha_weight).sum()
    classification = classification / positive_count
    if positive.any():
        decoded = decode_flat_boxes(flat, architecture)
        predicted_positive = decoded[positive]
        target_positive = target_boxes[positive]
        box_loss = (1.0 - aligned_ciou(predicted_positive, target_positive)).mean()
        batch_indices, anchor_indices = torch.nonzero(positive, as_tuple=True)
        anchors = flat["anchors"][anchor_indices]
        strides = flat["strides"][anchor_indices]
        distances = torch.stack(
            (
                anchors[:, 0] - target_positive[:, 0],
                anchors[:, 1] - target_positive[:, 1],
                target_positive[:, 2] - anchors[:, 0],
                target_positive[:, 3] - anchors[:, 1],
            ),
            dim=-1,
        ) / strides[:, None]
        reg_max = int(architecture["reg_max"])
        distances = distances.clamp(0.0, reg_max - 1.0001)
        lower = torch.floor(distances).long()
        upper = (lower + 1).clamp_max(reg_max - 1)
        upper_weight = distances - lower.to(distances.dtype)
        positive_logits = flat["box_logits"][batch_indices, anchor_indices].reshape(-1, 4, reg_max)
        log_probabilities = manual_log_softmax(positive_logits, -1)
        lower_log = log_probabilities.gather(-1, lower.unsqueeze(-1)).squeeze(-1)
        upper_log = log_probabilities.gather(-1, upper.unsqueeze(-1)).squeeze(-1)
        distribution_loss = -((1.0 - upper_weight) * lower_log + upper_weight * upper_log).mean()
    else:
        zero = flat["box_logits"].sum() * 0.0
        box_loss = zero
        distribution_loss = zero
    total = 7.5 * box_loss + 0.5 * classification + 1.5 * distribution_loss
    return total, {"box": box_loss, "class": classification, "distribution": distribution_loss}


class ManualAdamW:
    """Explicit AdamW state and tensor updates; no optimizer object is imported."""

    def __init__(self, named_parameters, beta1=0.9, beta2=0.999, epsilon=1e-8, weight_decay=5e-4):
        self.named = list(named_parameters)
        self.beta1 = float(beta1)
        self.beta2 = float(beta2)
        self.epsilon = float(epsilon)
        self.weight_decay = float(weight_decay)
        self.step_index = 0
        self.first = {name: torch.zeros_like(value) for name, value in self.named}
        self.second = {name: torch.zeros_like(value) for name, value in self.named}

    def zero_grad(self):
        for _, parameter in self.named:
            parameter.grad = None

    def step(self, learning_rate, max_gradient_norm=10.0):
        squared_norm = torch.zeros((), device=self.named[0][1].device)
        for _, parameter in self.named:
            if parameter.grad is not None:
                squared_norm = squared_norm + (parameter.grad.detach() ** 2).sum()
        norm = float(torch.sqrt(squared_norm).detach().cpu())
        gradient_scale = min(1.0, max_gradient_norm / (norm + 1e-12))
        self.step_index += 1
        correction1 = 1.0 - self.beta1 ** self.step_index
        correction2 = 1.0 - self.beta2 ** self.step_index
        with torch.no_grad():
            for name, parameter in self.named:
                if parameter.grad is None:
                    continue
                gradient = parameter.grad * gradient_scale
                self.first[name].mul_(self.beta1).add_((1.0 - self.beta1) * gradient)
                self.second[name].mul_(self.beta2).add_((1.0 - self.beta2) * (gradient * gradient))
                first_hat = self.first[name] / correction1
                second_hat = self.second[name] / correction2
                update = first_hat / (torch.sqrt(second_hat) + self.epsilon)
                parameter.add_(-learning_rate * (update + self.weight_decay * parameter))
        return norm

    def state_dict(self):
        return {
            "step_index": self.step_index,
            "beta1": self.beta1,
            "beta2": self.beta2,
            "epsilon": self.epsilon,
            "weight_decay": self.weight_decay,
            "first_moment": {name: value.detach().cpu() for name, value in self.first.items()},
            "second_moment": {name: value.detach().cpu() for name, value in self.second.items()},
        }

    def load_state_dict(self, state):
        self.step_index = int(state["step_index"])
        with torch.no_grad():
            for name, parameter in self.named:
                self.first[name].copy_(state["first_moment"][name].to(parameter.device))
                self.second[name].copy_(state["second_moment"][name].to(parameter.device))


def scheduled_learning_rate(step, total_steps, base_rate=2e-3, minimum_rate=2e-5, warmup_steps=100):
    if step < warmup_steps:
        return base_rate * float(step + 1) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    progress = min(1.0, max(0.0, progress))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return minimum_rate + (base_rate - minimum_rate) * cosine


def load_label_tensor(label_path, image_width, image_height):
    boxes, classes = [], []
    if Path(label_path).is_file():
        for raw_line in Path(label_path).read_text(encoding="utf-8-sig", errors="replace").splitlines():
            fields = raw_line.split()
            if len(fields) != 5:
                continue
            class_id, xc, yc, width, height = fields
            class_id = int(class_id)
            xc, yc, width, height = map(float, (xc, yc, width, height))
            boxes.append(
                [
                    (xc - width * 0.5) * image_width,
                    (yc - height * 0.5) * image_height,
                    (xc + width * 0.5) * image_width,
                    (yc + height * 0.5) * image_height,
                ]
            )
            classes.append(class_id)
    return boxes, classes


def prepare_pil_image(image, image_size, label_path=None, horizontal_flip=False):
    from PIL import Image

    image = image.convert("RGB")
    original_width, original_height = image.size
    scale = min(image_size / original_width, image_size / original_height)
    resized_width = max(1, round(original_width * scale))
    resized_height = max(1, round(original_height * scale))
    resized = image.resize((resized_width, resized_height), Image.Resampling.BILINEAR)
    pad_x = (image_size - resized_width) // 2
    pad_y = (image_size - resized_height) // 2
    canvas = Image.new("RGB", (image_size, image_size), (114, 114, 114))
    canvas.paste(resized, (pad_x, pad_y))
    boxes, classes = load_label_tensor(label_path, original_width, original_height) if label_path else ([], [])
    transformed = []
    for x1, y1, x2, y2 in boxes:
        transformed.append([x1 * scale + pad_x, y1 * scale + pad_y, x2 * scale + pad_x, y2 * scale + pad_y])
    if horizontal_flip:
        canvas = canvas.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        transformed = [[image_size - box[2], box[1], image_size - box[0], box[3]] for box in transformed]
    array = np.asarray(canvas, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()
    target = {
        "boxes": torch.tensor(transformed, dtype=torch.float32).reshape(-1, 4),
        "classes": torch.tensor(classes, dtype=torch.long),
    }
    transform = {
        "scale": scale,
        "pad_x": pad_x,
        "pad_y": pad_y,
        "original_width": original_width,
        "original_height": original_height,
    }
    return tensor, target, transform


def make_record_batches(records, batch_size, image_size, device, shuffle=False, augment=False, seed=0):
    from PIL import Image

    indices = list(range(len(records)))
    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(indices)
    for start in range(0, len(indices), batch_size):
        group = [records[index] for index in indices[start : start + batch_size]]
        images, targets, transforms = [], [], []
        for offset, record in enumerate(group):
            with Image.open(record["source_image_path"]) as image:
                flip = augment and random.Random(seed * 1000003 + start + offset).random() < 0.5
                tensor, target, transform = prepare_pil_image(
                    image, image_size, record["expected_label_path"], flip
                )
            images.append(tensor)
            targets.append({key: value.to(device) for key, value in target.items()})
            transforms.append(transform)
        yield torch.stack(images).to(device), targets, group, transforms


def manual_class_aware_nms(boxes, scores, classes, iou_threshold=0.6, max_detections=300):
    kept = []
    for class_id in torch.unique(classes).tolist():
        indices = torch.nonzero(classes == class_id, as_tuple=False).reshape(-1)
        indices = indices[torch.argsort(scores[indices], descending=True)]
        while indices.numel() and len(kept) < max_detections:
            current = int(indices[0].item())
            kept.append(current)
            if indices.numel() == 1:
                break
            remaining = indices[1:]
            overlap = pairwise_iou(boxes[current : current + 1], boxes[remaining]).reshape(-1)
            indices = remaining[overlap <= iou_threshold]
    if not kept:
        return torch.empty(0, dtype=torch.long, device=boxes.device)
    keep = torch.tensor(kept, dtype=torch.long, device=boxes.device)
    return keep[torch.argsort(scores[keep], descending=True)[:max_detections]]


def decode_predictions(outputs, architecture, confidence_threshold=0.25, iou_threshold=0.6, image_size=640, max_detections=300):
    flat = flatten_detector_outputs(outputs, architecture)
    boxes = decode_flat_boxes(flat, architecture).clamp(0.0, float(image_size))
    class_probabilities = manual_sigmoid(flat["class_logits"])
    scores, classes = class_probabilities.max(dim=-1)
    results = []
    for batch_index in range(boxes.shape[0]):
        mask = scores[batch_index] >= confidence_threshold
        selected_boxes = boxes[batch_index][mask]
        selected_scores = scores[batch_index][mask]
        selected_classes = classes[batch_index][mask]
        if selected_scores.numel() > 3000:
            order = torch.argsort(selected_scores, descending=True)[:3000]
            selected_boxes, selected_scores, selected_classes = (
                selected_boxes[order], selected_scores[order], selected_classes[order]
            )
        keep = manual_class_aware_nms(
            selected_boxes, selected_scores, selected_classes, iou_threshold, max_detections
        )
        if keep.numel():
            results.append(
                torch.cat(
                    (
                        selected_boxes[keep],
                        selected_scores[keep, None],
                        selected_classes[keep, None].to(selected_boxes.dtype),
                    ),
                    dim=1,
                )
            )
        else:
            results.append(torch.empty((0, 6), device=boxes.device))
    return results


def restore_boxes_to_original(detections, transform):
    restored = detections.detach().cpu().clone()
    if restored.numel():
        restored[:, [0, 2]] = (restored[:, [0, 2]] - transform["pad_x"]) / transform["scale"]
        restored[:, [1, 3]] = (restored[:, [1, 3]] - transform["pad_y"]) / transform["scale"]
        restored[:, [0, 2]] = restored[:, [0, 2]].clamp(0, transform["original_width"] - 1)
        restored[:, [1, 3]] = restored[:, [1, 3]].clamp(0, transform["original_height"] - 1)
    return restored


def _match_predictions(predictions, ground_truth, class_count, iou_threshold, confidence_threshold):
    tp = fp = fn = 0
    class_tp = [0] * class_count
    class_fp = [0] * class_count
    class_fn = [0] * class_count
    for prediction, target in zip(predictions, ground_truth):
        pred = prediction[prediction[:, 4] >= confidence_threshold]
        used = set()
        for row in pred[torch.argsort(pred[:, 4], descending=True)]:
            class_id = int(row[5].item())
            candidates = torch.nonzero(target["classes"] == class_id, as_tuple=False).reshape(-1)
            candidates = torch.tensor([int(x) for x in candidates.tolist() if int(x) not in used], dtype=torch.long)
            if candidates.numel():
                overlaps = pairwise_iou(row[:4].reshape(1, 4), target["boxes"][candidates]).reshape(-1)
                best_position = int(torch.argmax(overlaps).item())
                if float(overlaps[best_position]) >= iou_threshold:
                    used.add(int(candidates[best_position].item()))
                    tp += 1
                    class_tp[class_id] += 1
                    continue
            fp += 1
            class_fp[class_id] += 1
        for gt_index, class_id_tensor in enumerate(target["classes"]):
            if gt_index not in used:
                class_id = int(class_id_tensor.item())
                fn += 1
                class_fn[class_id] += 1
    return tp, fp, fn, class_tp, class_fp, class_fn


def _average_precision_for_class(predictions, ground_truth, class_id, iou_threshold):
    entries = []
    gt_count = 0
    targets_by_image = {}
    for image_index, target in enumerate(ground_truth):
        indices = torch.nonzero(target["classes"] == class_id, as_tuple=False).reshape(-1)
        targets_by_image[image_index] = target["boxes"][indices]
        gt_count += int(indices.numel())
        for row in predictions[image_index]:
            if int(row[5].item()) == class_id:
                entries.append((float(row[4].item()), image_index, row[:4]))
    if gt_count == 0:
        return None
    entries.sort(key=lambda item: item[0], reverse=True)
    matched = defaultdict(set)
    tp_values, fp_values = [], []
    for _, image_index, box in entries:
        target_boxes = targets_by_image[image_index]
        available = [index for index in range(target_boxes.shape[0]) if index not in matched[image_index]]
        is_true = False
        if available:
            available_tensor = torch.tensor(available, dtype=torch.long)
            overlaps = pairwise_iou(box.reshape(1, 4), target_boxes[available_tensor]).reshape(-1)
            best = int(torch.argmax(overlaps).item())
            if float(overlaps[best]) >= iou_threshold:
                matched[image_index].add(available[best])
                is_true = True
        tp_values.append(1.0 if is_true else 0.0)
        fp_values.append(0.0 if is_true else 1.0)
    if not entries:
        return 0.0
    cumulative_tp = torch.tensor(tp_values).cumsum(0)
    cumulative_fp = torch.tensor(fp_values).cumsum(0)
    recall = cumulative_tp / max(1, gt_count)
    precision = cumulative_tp / (cumulative_tp + cumulative_fp).clamp_min(1e-9)
    ap = 0.0
    for recall_level in torch.linspace(0.0, 1.0, 101):
        valid = precision[recall >= recall_level]
        ap += float(valid.max().item()) if valid.numel() else 0.0
    return ap / 101.0


def compute_detection_metrics(predictions, ground_truth, class_names):
    class_count = len(class_names)
    iou_thresholds = [0.50 + 0.05 * index for index in range(10)]
    ap_by_threshold = []
    per_class_ap50 = {}
    for threshold in iou_thresholds:
        class_values = []
        for class_id, class_name in enumerate(class_names):
            value = _average_precision_for_class(predictions, ground_truth, class_id, threshold)
            if threshold == 0.50:
                per_class_ap50[class_name] = value
            if value is not None:
                class_values.append(value)
        ap_by_threshold.append(sum(class_values) / len(class_values) if class_values else 0.0)
    tp, fp, fn, class_tp, class_fp, class_fn = _match_predictions(
        predictions, ground_truth, class_count, 0.50, 0.25
    )
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    # Match once in descending confidence order. Thresholding prefixes of this
    # truthful match sequence yields the complete confidence curves efficiently.
    scored_matches = []
    total_ground_truth = sum(int(item["classes"].numel()) for item in ground_truth)
    for prediction, target in zip(predictions, ground_truth):
        used = set()
        for row in prediction[torch.argsort(prediction[:, 4], descending=True)]:
            class_id = int(row[5].item())
            candidates = [
                index
                for index, target_class in enumerate(target["classes"].tolist())
                if int(target_class) == class_id and index not in used
            ]
            is_true = False
            if candidates:
                candidate_tensor = torch.tensor(candidates, dtype=torch.long)
                overlaps = pairwise_iou(
                    row[:4].reshape(1, 4), target["boxes"][candidate_tensor]
                ).reshape(-1)
                best_position = int(torch.argmax(overlaps).item())
                if float(overlaps[best_position]) >= 0.50:
                    used.add(candidates[best_position])
                    is_true = True
            scored_matches.append((float(row[4].item()), is_true))
    confidence_axis = [index / 100.0 for index in range(101)]
    precision_curve, recall_curve, f1_curve = [], [], []
    for confidence in confidence_axis:
        selected_matches = [is_true for score, is_true in scored_matches if score >= confidence]
        curve_tp = sum(selected_matches)
        curve_fp = len(selected_matches) - curve_tp
        curve_fn = total_ground_truth - curve_tp
        p = curve_tp / max(1, curve_tp + curve_fp)
        r = curve_tp / max(1, curve_tp + curve_fn)
        precision_curve.append(p)
        recall_curve.append(r)
        f1_curve.append(2.0 * p * r / max(1e-12, p + r))
    confusion = torch.zeros((class_count + 1, class_count + 1), dtype=torch.float64)
    background = class_count
    for prediction, target in zip(predictions, ground_truth):
        pred = prediction[prediction[:, 4] >= 0.25]
        pairs = []
        if pred.numel() and target["boxes"].numel():
            overlaps = pairwise_iou(target["boxes"], pred[:, :4])
            for gt_index in range(overlaps.shape[0]):
                for pred_index in range(overlaps.shape[1]):
                    if float(overlaps[gt_index, pred_index]) >= 0.50:
                        pairs.append((float(overlaps[gt_index, pred_index]), gt_index, pred_index))
        used_gt, used_pred = set(), set()
        for _, gt_index, pred_index in sorted(pairs, reverse=True):
            if gt_index in used_gt or pred_index in used_pred:
                continue
            used_gt.add(gt_index); used_pred.add(pred_index)
            confusion[int(target["classes"][gt_index]), int(pred[pred_index, 5])] += 1
        for gt_index, class_id in enumerate(target["classes"]):
            if gt_index not in used_gt:
                confusion[int(class_id), background] += 1
        for pred_index, row in enumerate(pred):
            if pred_index not in used_pred:
                confusion[background, int(row[5])] += 1
    per_class = {}
    for class_id, class_name in enumerate(class_names):
        p = class_tp[class_id] / max(1, class_tp[class_id] + class_fp[class_id])
        r = class_tp[class_id] / max(1, class_tp[class_id] + class_fn[class_id])
        per_class[class_name] = {"precision": p, "recall": r, "ap50": per_class_ap50[class_name]}
    return {
        "precision": precision,
        "recall": recall,
        "map50": ap_by_threshold[0],
        "map50_95": sum(ap_by_threshold) / len(ap_by_threshold),
        "iou_thresholds": iou_thresholds,
        "map_by_iou": ap_by_threshold,
        "per_class": per_class,
        "curves": {
            "confidence": confidence_axis,
            "precision": precision_curve,
            "recall": recall_curve,
            "f1": f1_curve,
            "pr_recall": list(reversed(recall_curve)),
            "pr_precision": list(reversed(precision_curve)),
        },
        "confusion_matrix": confusion.tolist(),
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def evaluate_detector(model, records, architecture, image_size, batch_size, device):
    model.eval()
    predictions, ground_truth = [], []
    loss_totals = {"box": 0.0, "class": 0.0, "distribution": 0.0}
    batch_count = 0
    with torch.no_grad():
        for images, targets, _, _ in make_record_batches(
            records, batch_size, image_size, device, shuffle=False, augment=False
        ):
            outputs = model(images)
            _, parts = detector_loss(outputs, targets, architecture)
            for key in loss_totals:
                loss_totals[key] += float(parts[key].detach().cpu())
            decoded = decode_predictions(outputs, architecture, 0.001, 0.60, image_size, 300)
            predictions.extend(item.detach().cpu() for item in decoded)
            ground_truth.extend(
                {"boxes": item["boxes"].detach().cpu(), "classes": item["classes"].detach().cpu()}
                for item in targets
            )
            batch_count += 1
    metrics = compute_detection_metrics(predictions, ground_truth, architecture["class_names"])
    metrics["losses"] = {key: value / max(1, batch_count) for key, value in loss_totals.items()}
    metrics["evaluated_images"] = len(records)
    return metrics, predictions, ground_truth


def plot_validation_artifacts(metrics, output_dir, class_names):
    import matplotlib.pyplot as plt

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    labels = list(class_names) + ["background"]
    matrix = np.asarray(metrics["confusion_matrix"], dtype=float)
    for filename, normalize in (("confusion_matrix.png", False), ("confusion_matrix_normalized.png", True)):
        values = matrix.copy()
        if normalize:
            values = values / np.maximum(values.sum(axis=1, keepdims=True), 1.0)
        fig, ax = plt.subplots(figsize=(9, 8))
        image = ax.imshow(values, cmap="Blues")
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
        ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
        ax.set_title("Normalized confusion matrix" if normalize else "Confusion matrix")
        fig.colorbar(image, ax=ax)
        fig.tight_layout(); fig.savefig(output_dir / filename, dpi=180); plt.close(fig)
    curves = metrics["curves"]
    curve_specs = [
        ("P_curve.png", curves["confidence"], curves["precision"], "Confidence", "Precision"),
        ("R_curve.png", curves["confidence"], curves["recall"], "Confidence", "Recall"),
        ("F1_curve.png", curves["confidence"], curves["f1"], "Confidence", "F1"),
        ("PR_curve.png", curves["pr_recall"], curves["pr_precision"], "Recall", "Precision"),
    ]
    for filename, x_values, y_values, x_label, y_label in curve_specs:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.plot(x_values, y_values, linewidth=2)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)
        ax.set_xlabel(x_label); ax.set_ylabel(y_label); ax.set_title(filename.replace("_", " ").replace(".png", ""))
        fig.tight_layout(); fig.savefig(output_dir / filename, dpi=180); plt.close(fig)


def checkpoint_payload(model, architecture, class_names, image_size, epoch, best_metric, seed, optimizer=None, training_state=None):
    payload = {
        "format": CUSTOM_CHECKPOINT_FORMAT,
        "format_version": CUSTOM_FORMAT_VERSION,
        "class_names": list(class_names),
        "class_count": len(class_names),
        "architecture": copy.deepcopy(architecture),
        "image_size": int(image_size),
        "model_state": model.state_dict(),
        "epoch": int(epoch),
        "best_validation_metric": float(best_metric),
        "random_seed": int(seed),
        "initialization": "random He initialization",
        "external_model_source": False,
        "prior_detector_weights_used": False,
    }
    if optimizer is not None:
        payload["optimizer_state"] = optimizer.state_dict()
    if training_state is not None:
        payload["training_state"] = copy.deepcopy(training_state)
    return payload


def safe_load_checkpoint(path, device="cpu"):
    try:
        checkpoint = torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        checkpoint = torch.load(path, map_location=device)
    if not isinstance(checkpoint, dict) or checkpoint.get("format") != CUSTOM_CHECKPOINT_FORMAT:
        raise ValueError(f"Not a {CUSTOM_CHECKPOINT_FORMAT} checkpoint: {path}")
    required = {"class_names", "class_count", "architecture", "image_size", "model_state", "epoch"}
    missing = sorted(required - set(checkpoint))
    if missing:
        raise ValueError(f"Checkpoint is missing required fields: {missing}")
    return checkpoint


def model_from_checkpoint(path, device):
    checkpoint = safe_load_checkpoint(path, device)
    model = CustomDetector(checkpoint["architecture"]).to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model, checkpoint


## 3. Dataset Scan and Validation

In [3]:
# Cell 3 - Scan source images and validate same-stem YOLO labels
REFERENCE_CLASS_REPORT = inspect_reference_classes([Path("classes.txt"), OUTPUT_ROOT.parent / "classes.txt"], CLASS_NAMES)
SCAN_RESULT = scan_dataset(IMAGE_DATASET_DIR, LABELS_ALL_DIR, OUTPUT_ROOT, CLASS_NAMES, BAD_IDS, TRAIN_ONLY_ID_RANGE)
print("Dataset scan completed.")
print(f"Source images found: {SCAN_RESULT['summary']['total_source_images_found']}")
print(f"Matching labels found: {SCAN_RESULT['summary']['total_matching_label_files_found']}")
print(f"Record status counts: {SCAN_RESULT['summary']['record_status_counts']}")
print("Dataset validation completed.")


Dataset scan completed.
Source images found: 3919
Matching labels found: 3919
Record status counts: {'valid_positive': 3506, 'excluded_bad_id': 2, 'empty_negative_label': 411}
Dataset validation completed.


## 4. Deterministic Train/Validation Split

In [4]:
# Cell 4 - Build deterministic train/val split only
SPLITS = create_train_val_split(SCAN_RESULT, CLASS_NAMES, BAD_IDS, ALLOWED_LEGACY_IDS, TRAIN_ONLY_ID_RANGE, RANDOM_SEED, VALIDATION_RATIO)
print("Train/validation split created.")
print(f"Train images: {SPLITS['summary']['train_images']}")
print(f"Validation images: {SPLITS['summary']['validation_images']}")
print(f"Actual train percentage: {SPLITS['summary']['final_train_percentage']:.2f}%")
print(f"Actual validation percentage: {SPLITS['summary']['final_validation_percentage']:.2f}%")

if SPLITS["summary"].get("validation_class_ids_unavailable"):
    unavailable = SPLITS["summary"]["validation_class_ids_unavailable"]
    names = SPLITS["summary"]["validation_class_names_unavailable"]
    print(f"Warning: validation has no eligible positive candidates for class IDs {unavailable}: {names}.")
if SPLITS["summary"].get("validation_candidate_class_ids_missing_from_val"):
    missing = SPLITS["summary"]["validation_candidate_class_ids_missing_from_val"]
    print(f"Warning: validation split could not represent candidate class IDs {missing} within the target validation size.")


Train/validation split created.
Train images: 3525
Validation images: 392
Actual train percentage: 89.99%
Actual validation percentage: 10.01%


## 5. Temporary Dataset and Audit Reports

In [5]:
# Cell 5 - Create /content train/val dataset and export audit reports
CONFIG_FOR_REPORTS = {
    "image_dataset_dir": IMAGE_DATASET_DIR,
    "labels_all_dir": LABELS_ALL_DIR,
    "output_root": OUTPUT_ROOT,
    "work_dir": WORK_DIR,
    "data_yaml_path": DATA_YAML_PATH,
    "model_format": CUSTOM_CHECKPOINT_FORMAT,
    "architecture": MODEL_CONFIG,
    "bad_ids": BAD_IDS,
    "allowed_legacy_ids": ALLOWED_LEGACY_IDS,
    "train_only_id_range": TRAIN_ONLY_ID_RANGE,
    "random_seed": RANDOM_SEED,
    "validation_ratio": VALIDATION_RATIO,
    "img_size": IMG_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "workers": WORKERS,
    "patience": PATIENCE,
    "cache": CACHE,
}
COPIED_MANIFEST = materialize_yolo_dataset(SPLITS, WORK_DIR, CLASS_NAMES, DATA_YAML_PATH, BAD_IDS, TRAIN_ONLY_ID_RANGE)
REPORT_PATHS = export_dataset_reports(SCAN_RESULT, SPLITS, COPIED_MANIFEST, OUTPUT_ROOT, CLASS_NAMES, CONFIG_FOR_REPORTS, REFERENCE_CLASS_REPORT)
print("Dataset reports exported.")
print(f"Temporary data.yaml: {DATA_YAML_PATH}")
print(f"Split manifest: {REPORT_PATHS['split_manifest_csv']}")
print(f"Dataset overview: {REPORT_PATHS['dataset_overview_json']}")


Dataset reports exported.
Temporary data.yaml: /content/uniform_yolo_dataset/data.yaml
Split manifest: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/split_manifest.csv
Dataset overview: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/dataset_overview.json


## 6. Bounding Box Preview Samples

In [6]:
# Cell 6 - Generate train/val bbox preview samples under OUTPUT_ROOT only
from PIL import Image, ImageDraw, ImageFont

CLASS_COLORS = [
    (180, 180, 180),
    (0, 174, 239),
    (100, 100, 100),
    (239, 68, 68),
    (168, 85, 247),
    (245, 158, 11),
]


def readable_text_color(background: tuple[int, int, int]) -> tuple[int, int, int]:
    red, green, blue = background
    luminance = 0.299 * red + 0.587 * green + 0.114 * blue
    return (0, 0, 0) if luminance > 160 else (255, 255, 255)


def yolo_box_to_xyxy(box: dict[str, Any], image_width: int, image_height: int) -> tuple[int, int, int, int]:
    x_center = float(box["x_center"])
    y_center = float(box["y_center"])
    width = float(box["width"])
    height = float(box["height"])
    x_min = round((x_center - width / 2.0) * image_width)
    y_min = round((y_center - height / 2.0) * image_height)
    x_max = round((x_center + width / 2.0) * image_width)
    y_max = round((y_center + height / 2.0) * image_height)
    return (max(0, min(image_width - 1, x_min)), max(0, min(image_height - 1, y_min)), max(0, min(image_width - 1, x_max)), max(0, min(image_height - 1, y_max)))


def load_preview_boxes(label_path: Path) -> list[dict[str, Any]]:
    validation = validate_yolo_label_file(label_path, CLASS_NAMES)
    return validation["boxes"] if validation["is_valid"] and not validation["is_empty"] else []


def draw_preview(record: dict[str, Any], output_path: Path) -> None:
    image_path = Path(record["source_image_path"])
    label_path = Path(record["expected_label_path"])
    boxes = load_preview_boxes(label_path)
    with Image.open(image_path) as original:
        image = original.convert("RGB")
        width, height = image.size
        draw = ImageDraw.Draw(image)
        font = ImageFont.load_default()
        line_width = max(2, min(8, round(min(width, height) / 220)))
        if not boxes:
            draw.text((16, 16), "negative sample: no boxes", fill=(17, 24, 39), font=font)
        for box in boxes:
            class_id = int(box["class_id"])
            color = CLASS_COLORS[class_id]
            x_min, y_min, x_max, y_max = yolo_box_to_xyxy(box, width, height)
            for offset in range(line_width):
                draw.rectangle((x_min - offset, y_min - offset, x_max + offset, y_max + offset), outline=color)
            label = CLASS_NAMES[class_id]
            text_bbox = draw.textbbox((0, 0), label, font=font)
            text_width = text_bbox[2] - text_bbox[0]
            text_height = text_bbox[3] - text_bbox[1]
            text_y = max(0, y_min - text_height - 8)
            draw.rectangle((x_min, text_y, x_min + text_width + 8, text_y + text_height + 6), fill=color)
            draw.text((x_min + 4, text_y + 3), label, fill=readable_text_color(color), font=font)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        save_kwargs = {"quality": 95} if output_path.suffix.lower() in {".jpg", ".jpeg"} else {}
        image.save(output_path, **save_kwargs)

preview_root = OUTPUT_ROOT / "bbox_preview_samples"
if preview_root.exists():
    shutil.rmtree(preview_root)
preview_root.mkdir(parents=True, exist_ok=True)
created_previews: list[Path] = []
for split_name in ["train", "val"]:
    records = list(SPLITS[split_name])
    positive_records = [record for record in records if record["object_count"] > 0]
    empty_records = [record for record in records if record["original_record_status"] == "empty_negative_label"]
    selected: list[dict[str, Any]] = []
    if split_name == "train" and empty_records:
        selected.append(min(empty_records, key=lambda record: stable_score(record, RANDOM_SEED)))
    remaining_slots = max(0, PREVIEW_SAMPLES_PER_SPLIT - len(selected))
    candidate_pool = positive_records or records
    selected_stems = {item["image_stem"] for item in selected}
    candidate_pool = [record for record in candidate_pool if record["image_stem"] not in selected_stems]
    candidate_pool.sort(key=lambda record: stable_score(record, RANDOM_SEED + len(split_name)))
    selected.extend(candidate_pool[:remaining_slots])
    for record in selected:
        output_path = preview_root / split_name / Path(record["source_image_path"]).name
        draw_preview(record, output_path)
        created_previews.append(output_path)
print("BBox preview samples exported.")
print(f"Preview images: {len(created_previews)}")
print(f"Preview folder: {preview_root}")


BBox preview samples exported.
Preview images: 16
Preview folder: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/bbox_preview_samples


## 7A. Restore and Validate a Custom Checkpoint

In [7]:
# Cell 7A - Restore the latest valid custom best.pt checkpoint from Google Drive
# Run this after Cells 1-6 when a runtime was disconnected; it never initializes from this file for a new training run.
RUNS_ROOT = OUTPUT_ROOT / "runs"
if not OUTPUT_ROOT.exists():
    raise FileNotFoundError(f"OUTPUT_ROOT does not exist: {OUTPUT_ROOT}")

best_files = sorted(
    (path for path in OUTPUT_ROOT.rglob("best.pt") if path.is_file() and path.stat().st_size > 0),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
valid_best_files = []
for path in best_files:
    try:
        checkpoint = safe_load_checkpoint(path, "cpu")
        if checkpoint["class_names"] == CLASS_NAMES and checkpoint["class_count"] == NUM_CLASSES:
            valid_best_files.append(path)
    except Exception as exc:
        print(f"Skipped incompatible checkpoint {path}: {exc}")
if not valid_best_files:
    raise FileNotFoundError("No valid custom best.pt checkpoint was found under OUTPUT_ROOT.")

BEST_PT = valid_best_files[0]
RUN_DIR = BEST_PT.parent.parent if BEST_PT.parent.name == "weights" else BEST_PT.parent
LAST_PT = RUN_DIR / "weights" / "last.pt"
if LAST_PT.exists():
    safe_load_checkpoint(LAST_PT, "cpu")
if not DATA_YAML_PATH.exists():
    raise FileNotFoundError("Run Cells 1-6 first so the temporary data.yaml is recreated.")
if "SPLITS" not in globals() or not SPLITS.get("val"):
    raise RuntimeError("Run Cells 1-6 first so the validation records are available.")
print("Custom checkpoint restored successfully.")
print(f"BEST_PT: {BEST_PT}")
print(f"RUN_DIR: {RUN_DIR}")
print(f"Validation images restored: {len(SPLITS['val'])}")


Best checkpoints found:
  /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/best.pt
    Size: 21.48 MB | Modified: 1782783559.0

Last checkpoints found:
  /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/last.pt
    Size: 21.48 MB | Modified: 1782783559.0

Checkpoint restored successfully.
BEST_PT: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/best.pt
RUN_DIR: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training
Validation images restored: 392
You can now run the fixed Cell 8 without retraining.


## 7. Train the Custom Detector from Random Initialization

In [ ]:
# Cell 7 - Real custom training loop with manual tensor updates
if not DATA_YAML_PATH.exists():
    raise FileNotFoundError(f"Generated data.yaml not found: {DATA_YAML_PATH}")
if not SPLITS.get("train") or not SPLITS.get("val"):
    raise RuntimeError("Both train and validation records are required.")

seed_everything(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARCHITECTURE_CONFIG = copy.deepcopy(MODEL_CONFIG)
model = CustomDetector(ARCHITECTURE_CONFIG).to(DEVICE)
optimizer = ManualAdamW(model.named_parameters(), weight_decay=5e-4)
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_NAME
WEIGHTS_DIR = RUN_DIR / "weights"
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_PT = WEIGHTS_DIR / "best.pt"
LAST_PT = WEIGHTS_DIR / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

steps_per_epoch = math.ceil(len(SPLITS["train"]) / BATCH_SIZE)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = min(total_steps, max(100, 3 * steps_per_epoch))
history = []
best_metric = -1.0
epochs_without_improvement = 0
global_step = 0
training_started = time.time()
print("Training started from random He initialization.")

with quiet_execution(FRAMEWORK_LOG_DIR, "training") as TRAINING_LOG_PATH:
    for epoch in range(EPOCHS):
        model.train(True)
        running = {"box": 0.0, "class": 0.0, "distribution": 0.0}
        batch_count = 0
        last_lr = 0.0
        for images, targets, _, _ in make_record_batches(
            SPLITS["train"], BATCH_SIZE, IMG_SIZE, DEVICE, shuffle=True, augment=True, seed=RANDOM_SEED + epoch
        ):
            optimizer.zero_grad()
            outputs = model(images)
            total_loss, parts = detector_loss(outputs, targets, ARCHITECTURE_CONFIG)
            if not torch.isfinite(total_loss):
                raise FloatingPointError(f"Non-finite training loss at epoch {epoch + 1}, step {global_step + 1}")
            total_loss.backward()
            last_lr = scheduled_learning_rate(global_step, total_steps, warmup_steps=warmup_steps)
            gradient_norm = optimizer.step(last_lr)
            for key in running:
                running[key] += float(parts[key].detach().cpu())
            batch_count += 1
            global_step += 1

        validation_metrics, _, _ = evaluate_detector(
            model, SPLITS["val"], ARCHITECTURE_CONFIG, IMG_SIZE, BATCH_SIZE, DEVICE
        )
        row = {
            "epoch": epoch + 1,
            "time": time.time() - training_started,
            "train/box_loss": running["box"] / max(1, batch_count),
            "train/cls_loss": running["class"] / max(1, batch_count),
            "train/dfl_loss": running["distribution"] / max(1, batch_count),
            "metrics/precision(B)": validation_metrics["precision"],
            "metrics/recall(B)": validation_metrics["recall"],
            "metrics/mAP50(B)": validation_metrics["map50"],
            "metrics/mAP50-95(B)": validation_metrics["map50_95"],
            "val/box_loss": validation_metrics["losses"]["box"],
            "val/cls_loss": validation_metrics["losses"]["class"],
            "val/dfl_loss": validation_metrics["losses"]["distribution"],
            "lr/pg0": last_lr,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(RESULTS_CSV, index=False)
        training_state = {"global_step": global_step, "epochs_without_improvement": epochs_without_improvement, "history_rows": len(history)}
        torch.save(
            checkpoint_payload(model, ARCHITECTURE_CONFIG, CLASS_NAMES, IMG_SIZE, epoch + 1, max(best_metric, row["metrics/mAP50-95(B)"]), RANDOM_SEED, optimizer, training_state),
            LAST_PT,
        )
        if row["metrics/mAP50-95(B)"] > best_metric:
            best_metric = row["metrics/mAP50-95(B)"]
            epochs_without_improvement = 0
            torch.save(
                checkpoint_payload(model, ARCHITECTURE_CONFIG, CLASS_NAMES, IMG_SIZE, epoch + 1, best_metric, RANDOM_SEED),
                BEST_PT,
            )
        else:
            epochs_without_improvement += 1
        print(
            f"epoch {epoch + 1}/{EPOCHS}: loss={float(total_loss.detach().cpu()):.4f}, "
            f"mAP50={validation_metrics['map50']:.4f}, mAP50-95={validation_metrics['map50_95']:.4f}, "
            f"lr={last_lr:.6g}, grad_norm={gradient_norm:.3f}"
        )
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without validation improvement.")
            break

training_args = {
    "model_format": CUSTOM_CHECKPOINT_FORMAT,
    "architecture": ARCHITECTURE_CONFIG,
    "random_initialization": True,
    "external_model_source": False,
    "prior_detector_weights_used": False,
    "data": str(DATA_YAML_PATH),
    "imgsz": IMG_SIZE,
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "workers": WORKERS,
    "patience": PATIENCE,
    "cache": CACHE,
    "seed": RANDOM_SEED,
    "optimizer": "manual AdamW tensor update",
    "scheduler": "manual warmup plus cosine",
}
(RUN_DIR / "training_args.yaml").write_text(yaml.safe_dump(training_args, sort_keys=False), encoding="utf-8")
shutil.copy2(RESULTS_CSV, OUTPUT_ROOT / "results.csv")
shutil.copy2(RUN_DIR / "training_args.yaml", OUTPUT_ROOT / "training_args.yaml")
TRAINING_CURVES_PATH = export_training_curves(RESULTS_CSV, OUTPUT_ROOT / "training_curves.png")
TRAINING_RESULTS_SUMMARY = summarize_results_csv(RESULTS_CSV)
TRAIN_RESULTS = {"history": history, "best_metric": best_metric}
write_json(
    OUTPUT_ROOT / "training_run_metadata.json",
    {
        "run_dir": str(RUN_DIR),
        "best_pt": str(BEST_PT),
        "last_pt": str(LAST_PT),
        "results_csv": str(RESULTS_CSV),
        "training_log": str(TRAINING_LOG_PATH),
        "training_results_summary": TRAINING_RESULTS_SUMMARY,
        "model_format": CUSTOM_CHECKPOINT_FORMAT,
        "random_initialization": True,
        "external_model_source": False,
        "prior_detector_weights_used": False,
    },
)
print("Training completed.")
print(f"Run directory: {RUN_DIR}")
print(f"Raw training log: {TRAINING_LOG_PATH}")
for key, value in TRAINING_RESULTS_SUMMARY.items():
    print(f"{key}: {value}")


Training started.
Best checkpoint verified.
Training completed.
Run directory: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training
Raw training log: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/framework_logs/20260629_202403_training.log
rows: 152
metrics/precision(B): 0.93354
metrics/recall(B): 0.94396
metrics/mAP50(B): 0.95761
metrics/mAP50-95(B): 0.73609


## 8. Manual Validation Metrics and Curves

In [8]:
# Cell 8 - Validate the custom best.pt and export metrics computed from its real predictions
if not Path(BEST_PT).is_file():
    raise FileNotFoundError(f"BEST_PT is unavailable: {BEST_PT}")
if not SPLITS.get("val"):
    raise RuntimeError("Validation split is unavailable.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
validation_model, validation_checkpoint = model_from_checkpoint(BEST_PT, DEVICE)
if validation_checkpoint["class_names"] != CLASS_NAMES:
    raise ValueError("Checkpoint class order does not match the required six-class order.")
VAL_RUN_DIR = OUTPUT_ROOT / "runs" / VALIDATION_RUN_NAME
VAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Validation started.")
with quiet_execution(FRAMEWORK_LOG_DIR, "validation") as VALIDATION_LOG_PATH:
    VALIDATION_METRICS, VALIDATION_PREDICTIONS, VALIDATION_TARGETS = evaluate_detector(
        validation_model, SPLITS["val"], validation_checkpoint["architecture"], IMG_SIZE, BATCH_SIZE, DEVICE
    )
    plot_validation_artifacts(VALIDATION_METRICS, VAL_RUN_DIR, CLASS_NAMES)

required_plots = [
    "confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png",
    "F1_curve.png", "P_curve.png", "R_curve.png",
]
for filename in required_plots:
    source = VAL_RUN_DIR / filename
    if not source.is_file():
        raise FileNotFoundError(f"Validation did not create {filename}")
    shutil.copy2(source, OUTPUT_ROOT / filename)

validation_class_counts = object_counts_by_class(SPLITS["val"], CLASS_NAMES)
metrics_payload = {
    "metrics": {
        "metrics/precision(B)": VALIDATION_METRICS["precision"],
        "metrics/recall(B)": VALIDATION_METRICS["recall"],
        "metrics/mAP50(B)": VALIDATION_METRICS["map50"],
        "metrics/mAP50-95(B)": VALIDATION_METRICS["map50_95"],
        "tp": VALIDATION_METRICS["tp"],
        "fp": VALIDATION_METRICS["fp"],
        "fn": VALIDATION_METRICS["fn"],
        "per_class": VALIDATION_METRICS["per_class"],
    },
    "nc": NUM_CLASSES,
    "class_names": {index: name for index, name in enumerate(CLASS_NAMES)},
    "validation_object_instances_per_class": validation_class_counts,
    "validation_class_ids_present": sorted({class_id for record in SPLITS["val"] for class_id in record["class_ids_present"]}),
    "validation_executed": True,
    "model_format": CUSTOM_CHECKPOINT_FORMAT,
    "warnings": [],
    "errors": [],
}
write_json(OUTPUT_ROOT / "validation_metrics.json", metrics_payload)
summary_rows = [
    {"row_type": "metric", "metric": "Precision", "value": VALIDATION_METRICS["precision"], "class_id": "", "class_name": "", "instances": ""},
    {"row_type": "metric", "metric": "Recall", "value": VALIDATION_METRICS["recall"], "class_id": "", "class_name": "", "instances": ""},
    {"row_type": "metric", "metric": "mAP@0.50", "value": VALIDATION_METRICS["map50"], "class_id": "", "class_name": "", "instances": ""},
    {"row_type": "metric", "metric": "mAP@0.50:0.95", "value": VALIDATION_METRICS["map50_95"], "class_id": "", "class_name": "", "instances": ""},
]
summary_rows.extend(
    {"row_type": "validation_class_instances", "metric": "object_instances", "value": "", "class_id": class_id, "class_name": class_name, "instances": validation_class_counts[class_name]}
    for class_id, class_name in enumerate(CLASS_NAMES)
)
pd.DataFrame(summary_rows).to_csv(OUTPUT_ROOT / "validation_summary.csv", index=False, encoding="utf-8-sig")
artifact_report = {
    "validation_executed": True,
    "validation_started_at": datetime.now().isoformat(timespec="seconds"),
    "validation_run_dir": str(VAL_RUN_DIR),
    "validation_log": str(VALIDATION_LOG_PATH),
    "artifact_status": {name: {"status": "generated_from_custom_predictions", "source": str(VAL_RUN_DIR / name)} for name in required_plots},
    "artifact_generation": {name: "manual evaluator output" for name in required_plots},
    "artifact_sources": {name: str(VAL_RUN_DIR / name) for name in required_plots},
    "candidate_artifact_directories": [str(VAL_RUN_DIR)],
    "warnings": [],
    "errors": [],
}
write_json(OUTPUT_ROOT / "validation_artifact_report.json", artifact_report)
write_json(
    OUTPUT_ROOT / "validation_run_metadata.json",
    {
        "validated_split": "val",
        "validation_executed": True,
        "validation_run_dir": str(VAL_RUN_DIR),
        "validation_log": str(VALIDATION_LOG_PATH),
        "validation_metrics_json": str(OUTPUT_ROOT / "validation_metrics.json"),
        "validation_summary_csv": str(OUTPUT_ROOT / "validation_summary.csv"),
        "validation_artifact_report_json": str(OUTPUT_ROOT / "validation_artifact_report.json"),
        "warnings_count": 0,
        "errors_count": 0,
    },
)
VAL_RESULTS = VALIDATION_METRICS
print("Validation completed.")
print(f"Precision: {VALIDATION_METRICS['precision']:.6f}")
print(f"Recall: {VALIDATION_METRICS['recall']:.6f}")
print(f"mAP@0.50: {VALIDATION_METRICS['map50']:.6f}")
print(f"mAP@0.50:0.95: {VALIDATION_METRICS['map50_95']:.6f}")
print(f"Validation artifacts: {VAL_RUN_DIR}")


Validation started.
Validation completed.
Raw validation log: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/framework_logs/20260630_053824_validation.log
Precision: 0.9319
Recall: 0.9373
mAP@0.50: 0.9574
mAP@0.50:0.95: 0.7376
All required validation artifacts are available.
Metrics exported.


## 9. Custom Checkpoint Sample Inference

In [9]:
# Cell 9 - Run concise sample inference, preferring validation positives
from PIL import Image, ImageDraw, ImageFont

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model, inference_checkpoint = model_from_checkpoint(BEST_PT, DEVICE)
sample_output_dir = OUTPUT_ROOT / "sample_predictions"
if sample_output_dir.exists():
    shutil.rmtree(sample_output_dir)
sample_output_dir.mkdir(parents=True, exist_ok=True)
sample_candidates = [record for record in SPLITS["val"] if record["object_count"] > 0] or list(SPLITS["val"])
sample_candidates.sort(key=lambda record: stable_score(record, RANDOM_SEED + 909))
selected_samples = sample_candidates[:SAMPLE_PREDICTION_COUNT]
created_predictions = []
print("Sample inference started.")
with quiet_execution(FRAMEWORK_LOG_DIR, "sample_inference") as SAMPLE_INFERENCE_LOG_PATH:
    for sample_index, record in enumerate(selected_samples):
        with Image.open(record["source_image_path"]) as source_image:
            original = source_image.convert("RGB")
            tensor, _, transform = prepare_pil_image(original, IMG_SIZE)
        with torch.no_grad():
            outputs = inference_model(tensor.unsqueeze(0).to(DEVICE))
            detections = decode_predictions(
                outputs, inference_checkpoint["architecture"], 0.25, 0.60, IMG_SIZE, 300
            )[0]
        detections = restore_boxes_to_original(detections, transform)
        draw = ImageDraw.Draw(original)
        font = ImageFont.load_default()
        for row in detections:
            x1, y1, x2, y2, confidence, class_value = row.tolist()
            class_id = int(class_value)
            color = CLASS_COLORS[class_id]
            draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
            draw.text((x1 + 2, max(0, y1 - 14)), f"{CLASS_NAMES[class_id]} {confidence:.3f}", fill=color, font=font)
        output_path = sample_output_dir / f"image{sample_index}.jpg"
        original.save(output_path, quality=95)
        created_predictions.append(output_path)
        print(f"{Path(record['source_image_path']).name}: {len(detections)} detections")
        for row in detections:
            x1, y1, x2, y2, confidence, class_value = row.tolist()
            print(f"  {CLASS_NAMES[int(class_value)]}: conf={confidence:.3f}, box_xyxy={[round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)]}")
print("Sample inference completed.")
print(f"Annotated predictions: {len(created_predictions)}")
print(f"Prediction folder: {sample_output_dir}")
print(f"Raw inference log: {SAMPLE_INFERENCE_LOG_PATH}")


Sample inference started.
image0.jpg: 3 detections
  quan_tay_dai_den: 0.890
  ao_so_mi_trang: 0.885
  khan_quang_do: 0.878
image1.jpg: 3 detections
  ao_doan_thanh_nien: 0.921
  quan_dai_trang: 0.899
  khan_quang_do: 0.845
image2.jpg: 2 detections
  quan_tay_dai_den: 0.912
  ao_doan_thanh_nien: 0.881
image3.jpg: 1 detections
  quan_dai_trang: 0.918
image4.jpg: 3 detections
  ao_so_mi_trang: 0.748
  quan_dai_trang: 0.671
  ao_so_mi_trang: 0.452
image5.jpg: 4 detections
  quan_short_tay_den: 0.928
  ao_so_mi_trang: 0.817
  khan_quang_do: 0.404
  khan_quang_do: 0.340
image6.jpg: 2 detections
  ao_doan_thanh_nien: 0.908
  quan_tay_dai_den: 0.874
image7.jpg: 3 detections
  ao_so_mi_trang: 0.907
  khan_quang_do: 0.820
  quan_tay_dai_den: 0.560
Sample predictions saved to: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/sample_predictions
Raw inference log: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/framework_logs/20260630_053858_sample_inference.log


## 10. Self-Contained Windows 11 Deployment Package

In [11]:
# Cell 10 - Build the Windows 11 package for the custom checkpoint
import zipfile

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

INFER_WINDOWS_SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport copy\nimport math\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\nCUSTOM_CHECKPOINT_FORMAT = "custom_anchor_free_detector_v1"\nCUSTOM_FORMAT_VERSION = 1\n\ndef manual_sigmoid(x):\n    """Numerically stable sigmoid written directly from exp and division."""\n    z = x.clamp(-60.0, 60.0)\n    return 1.0 / (1.0 + torch.exp(-z))\n\ndef manual_silu(x):\n    return x * manual_sigmoid(x)\n\ndef manual_softmax(x, dim=-1):\n    shifted = x - x.amax(dim=dim, keepdim=True)\n    numerator = torch.exp(shifted)\n    return numerator / numerator.sum(dim=dim, keepdim=True).clamp_min(1e-12)\n\ndef _component_children(value):\n    if isinstance(value, TensorComponent):\n        yield value\n    elif isinstance(value, (list, tuple)):\n        for item in value:\n            yield from _component_children(item)\n    elif isinstance(value, dict):\n        for item in value.values():\n            yield from _component_children(item)\n\nclass TensorComponent:\n    """Small recursive tensor container; all forward mathematics lives below."""\n\n    def __init__(self):\n        self.training = True\n\n    def named_parameters(self, prefix=""):\n        for name, value in self.__dict__.items():\n            full = f"{prefix}.{name}" if prefix else name\n            if isinstance(value, torch.Tensor) and value.requires_grad:\n                yield full, value\n            elif isinstance(value, TensorComponent):\n                yield from value.named_parameters(full)\n            elif isinstance(value, (list, tuple)):\n                for index, item in enumerate(value):\n                    if isinstance(item, TensorComponent):\n                        yield from item.named_parameters(f"{full}.{index}")\n\n    def named_tensors(self, prefix=""):\n        for name, value in self.__dict__.items():\n            full = f"{prefix}.{name}" if prefix else name\n            if isinstance(value, torch.Tensor):\n                yield full, value\n            elif isinstance(value, TensorComponent):\n                yield from value.named_tensors(full)\n            elif isinstance(value, (list, tuple)):\n                for index, item in enumerate(value):\n                    if isinstance(item, TensorComponent):\n                        yield from item.named_tensors(f"{full}.{index}")\n\n    def parameters(self):\n        return [value for _, value in self.named_parameters()]\n\n    def state_dict(self):\n        return {name: value.detach().cpu().clone() for name, value in self.named_tensors()}\n\n    def load_state_dict(self, state):\n        current = dict(self.named_tensors())\n        missing = sorted(set(current) - set(state))\n        unexpected = sorted(set(state) - set(current))\n        if missing or unexpected:\n            raise ValueError(f"State mismatch; missing={missing[:5]}, unexpected={unexpected[:5]}")\n        with torch.no_grad():\n            for name, value in current.items():\n                incoming = state[name].to(device=value.device, dtype=value.dtype)\n                if incoming.shape != value.shape:\n                    raise ValueError(f"Shape mismatch for {name}: {incoming.shape} != {value.shape}")\n                value.copy_(incoming)\n        return self\n\n    def to(self, device):\n        for name, value in list(self.__dict__.items()):\n            if isinstance(value, torch.Tensor):\n                needs_grad = value.requires_grad\n                moved = value.detach().to(device).requires_grad_(needs_grad)\n                setattr(self, name, moved)\n            elif isinstance(value, TensorComponent):\n                value.to(device)\n            elif isinstance(value, list):\n                for item in value:\n                    if isinstance(item, TensorComponent):\n                        item.to(device)\n        return self\n\n    def train(self, mode=True):\n        self.training = bool(mode)\n        for value in self.__dict__.values():\n            for child in _component_children(value):\n                child.train(mode)\n        return self\n\n    def eval(self):\n        return self.train(False)\n\n    def __call__(self, *args, **kwargs):\n        return self.forward(*args, **kwargs)\n\ndef he_tensor(shape, fan_in):\n    """Explicit He/Kaiming uniform initialization from random tensor arithmetic."""\n    bound = math.sqrt(6.0 / max(1, fan_in))\n    return ((torch.rand(*shape) * 2.0 - 1.0) * bound).requires_grad_(True)\n\ndef manual_pad_2d(x, padding):\n    if padding == 0:\n        return x\n    batch, channels, height, width = x.shape\n    side = torch.zeros(batch, channels, height, padding, device=x.device, dtype=x.dtype)\n    x = torch.cat((side, x, side), dim=3)\n    top = torch.zeros(batch, channels, padding, width + 2 * padding, device=x.device, dtype=x.dtype)\n    return torch.cat((top, x, top), dim=2)\n\nclass ManualConv(TensorComponent):\n    """Convolution explicitly formed as image patches multiplied by learned kernels."""\n\n    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=None, bias=True):\n        super().__init__()\n        self.in_channels = int(in_channels)\n        self.out_channels = int(out_channels)\n        self.kernel_size = int(kernel_size)\n        self.stride = int(stride)\n        self.padding = self.kernel_size // 2 if padding is None else int(padding)\n        fan_in = self.in_channels * self.kernel_size * self.kernel_size\n        self.weight = he_tensor(\n            (self.out_channels, self.in_channels, self.kernel_size, self.kernel_size), fan_in\n        )\n        self.bias = torch.zeros(self.out_channels, requires_grad=True) if bias else None\n\n    def forward(self, x):\n        if x.ndim != 4 or x.shape[1] != self.in_channels:\n            raise ValueError(f"Expected NCHW with {self.in_channels} channels, got {tuple(x.shape)}")\n        padded = manual_pad_2d(x, self.padding)\n        patches = padded.unfold(2, self.kernel_size, self.stride).unfold(\n            3, self.kernel_size, self.stride\n        )\n        batch, _, out_h, out_w, _, _ = patches.shape\n        rows = patches.permute(0, 2, 3, 1, 4, 5).reshape(batch * out_h * out_w, -1)\n        kernels = self.weight.reshape(self.out_channels, -1)\n        result = rows @ kernels.transpose(0, 1)\n        if self.bias is not None:\n            result = result + self.bias.reshape(1, -1)\n        return result.reshape(batch, out_h, out_w, self.out_channels).permute(0, 3, 1, 2)\n\nclass ManualBatchNorm(TensorComponent):\n    def __init__(self, channels, momentum=0.03, epsilon=1e-3):\n        super().__init__()\n        self.channels = int(channels)\n        self.momentum = float(momentum)\n        self.epsilon = float(epsilon)\n        self.scale = torch.ones(channels, requires_grad=True)\n        self.shift = torch.zeros(channels, requires_grad=True)\n        self.running_mean = torch.zeros(channels)\n        self.running_variance = torch.ones(channels)\n\n    def forward(self, x):\n        if self.training:\n            mean = x.mean(dim=(0, 2, 3))\n            centered = x - mean.reshape(1, -1, 1, 1)\n            variance = (centered * centered).mean(dim=(0, 2, 3))\n            with torch.no_grad():\n                self.running_mean.mul_(1.0 - self.momentum).add_(self.momentum * mean.detach())\n                self.running_variance.mul_(1.0 - self.momentum).add_(\n                    self.momentum * variance.detach()\n                )\n        else:\n            mean = self.running_mean\n            variance = self.running_variance\n            centered = x - mean.reshape(1, -1, 1, 1)\n        normalized = centered / torch.sqrt(variance.reshape(1, -1, 1, 1) + self.epsilon)\n        return normalized * self.scale.reshape(1, -1, 1, 1) + self.shift.reshape(1, -1, 1, 1)\n\nclass ConvNormAct(TensorComponent):\n    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):\n        super().__init__()\n        self.conv = ManualConv(in_channels, out_channels, kernel_size, stride, bias=False)\n        self.norm = ManualBatchNorm(out_channels)\n\n    def forward(self, x):\n        return manual_silu(self.norm(self.conv(x)))\n\nclass Bottleneck(TensorComponent):\n    def __init__(self, channels, shortcut=True):\n        super().__init__()\n        self.first = ConvNormAct(channels, channels, 3, 1)\n        self.second = ConvNormAct(channels, channels, 3, 1)\n        self.shortcut = bool(shortcut)\n\n    def forward(self, x):\n        y = self.second(self.first(x))\n        return x + y if self.shortcut else y\n\nclass C2f(TensorComponent):\n    def __init__(self, in_channels, out_channels, repeats=1):\n        super().__init__()\n        self.hidden = max(1, out_channels // 2)\n        self.entry = ConvNormAct(in_channels, self.hidden * 2, 1, 1)\n        self.blocks = [Bottleneck(self.hidden, True) for _ in range(int(repeats))]\n        self.exit = ConvNormAct(self.hidden * (2 + len(self.blocks)), out_channels, 1, 1)\n\n    def forward(self, x):\n        split = self.entry(x)\n        pieces = [split[:, : self.hidden], split[:, self.hidden :]]\n        for block in self.blocks:\n            pieces.append(block(pieces[-1]))\n        return self.exit(torch.cat(pieces, dim=1))\n\ndef manual_max_pool_same(x, kernel_size=5):\n    padding = kernel_size // 2\n    batch, channels, height, width = x.shape\n    side = torch.full(\n        (batch, channels, height, padding), -float("inf"), device=x.device, dtype=x.dtype\n    )\n    padded = torch.cat((side, x, side), dim=3)\n    top = torch.full(\n        (batch, channels, padding, width + 2 * padding),\n        -float("inf"),\n        device=x.device,\n        dtype=x.dtype,\n    )\n    padded = torch.cat((top, padded, top), dim=2)\n    windows = padded.unfold(2, kernel_size, 1).unfold(3, kernel_size, 1)\n    return windows.amax(dim=(-1, -2))\n\nclass SPPF(TensorComponent):\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n        hidden = max(1, in_channels // 2)\n        self.entry = ConvNormAct(in_channels, hidden, 1, 1)\n        self.exit = ConvNormAct(hidden * 4, out_channels, 1, 1)\n\n    def forward(self, x):\n        first = self.entry(x)\n        second = manual_max_pool_same(first, 5)\n        third = manual_max_pool_same(second, 5)\n        fourth = manual_max_pool_same(third, 5)\n        return self.exit(torch.cat((first, second, third, fourth), dim=1))\n\ndef manual_nearest_upsample(x, scale=2):\n    return x.repeat_interleave(scale, dim=2).repeat_interleave(scale, dim=3)\n\nclass DetectionHead(TensorComponent):\n    def __init__(self, in_channels, hidden, class_count, reg_max):\n        super().__init__()\n        self.box_stem = ConvNormAct(in_channels, hidden, 3, 1)\n        self.class_stem = ConvNormAct(in_channels, hidden, 3, 1)\n        self.box_output = ManualConv(hidden, 4 * reg_max, 1, 1, 0, True)\n        self.class_output = ManualConv(hidden, class_count, 1, 1, 0, True)\n        with torch.no_grad():\n            self.class_output.bias.fill_(-4.0)\n\n    def forward(self, x):\n        return torch.cat((self.box_output(self.box_stem(x)), self.class_output(self.class_stem(x))), dim=1)\n\nclass CustomDetector(TensorComponent):\n    """Compact C2f/SPPF backbone with a PAN/FPN neck and P3/P4/P5 heads."""\n\n    def __init__(self, architecture):\n        super().__init__()\n        self.architecture = copy.deepcopy(architecture)\n        channels = list(architecture["channels"])\n        repeats = list(architecture["repeats"])\n        class_count = int(architecture["class_count"])\n        reg_max = int(architecture["reg_max"])\n        c1, c2, c3, c4, c5 = channels\n        self.stem = ConvNormAct(3, c1, 3, 2)\n        self.down2 = ConvNormAct(c1, c2, 3, 2)\n        self.stage2 = C2f(c2, c2, repeats[0])\n        self.down3 = ConvNormAct(c2, c3, 3, 2)\n        self.stage3 = C2f(c3, c3, repeats[1])\n        self.down4 = ConvNormAct(c3, c4, 3, 2)\n        self.stage4 = C2f(c4, c4, repeats[2])\n        self.down5 = ConvNormAct(c4, c5, 3, 2)\n        self.stage5 = C2f(c5, c5, repeats[3])\n        self.sppf = SPPF(c5, c5)\n        self.reduce5 = ConvNormAct(c5, c4, 1, 1)\n        self.fuse4 = C2f(c4 + c4, c4, 1)\n        self.reduce4 = ConvNormAct(c4, c3, 1, 1)\n        self.fuse3 = C2f(c3 + c3, c3, 1)\n        self.neck_down4 = ConvNormAct(c3, c3, 3, 2)\n        self.pan4 = C2f(c3 + c4, c4, 1)\n        self.neck_down5 = ConvNormAct(c4, c4, 3, 2)\n        self.pan5 = C2f(c4 + c5, c5, 1)\n        self.head3 = DetectionHead(c3, c3, class_count, reg_max)\n        self.head4 = DetectionHead(c4, c4, class_count, reg_max)\n        self.head5 = DetectionHead(c5, c5, class_count, reg_max)\n\n    def forward(self, x):\n        x1 = self.stem(x)\n        x2 = self.stage2(self.down2(x1))\n        p3 = self.stage3(self.down3(x2))\n        p4 = self.stage4(self.down4(p3))\n        p5 = self.sppf(self.stage5(self.down5(p4)))\n        n4 = self.fuse4(torch.cat((manual_nearest_upsample(self.reduce5(p5)), p4), dim=1))\n        n3 = self.fuse3(torch.cat((manual_nearest_upsample(self.reduce4(n4)), p3), dim=1))\n        o4 = self.pan4(torch.cat((self.neck_down4(n3), n4), dim=1))\n        o5 = self.pan5(torch.cat((self.neck_down5(o4), p5), dim=1))\n        return [self.head3(n3), self.head4(o4), self.head5(o5)]\n\ndef flatten_detector_outputs(outputs, architecture):\n    reg_max = int(architecture["reg_max"])\n    class_count = int(architecture["class_count"])\n    box_parts, class_parts, anchor_parts, stride_parts, level_parts = [], [], [], [], []\n    for level, (output, stride) in enumerate(zip(outputs, architecture["strides"])):\n        batch, _, height, width = output.shape\n        flat = output.permute(0, 2, 3, 1).reshape(batch, height * width, 4 * reg_max + class_count)\n        box_parts.append(flat[:, :, : 4 * reg_max])\n        class_parts.append(flat[:, :, 4 * reg_max :])\n        yy, xx = torch.meshgrid(\n            torch.arange(height, device=output.device, dtype=output.dtype),\n            torch.arange(width, device=output.device, dtype=output.dtype),\n            indexing="ij",\n        )\n        anchor_parts.append(torch.stack(((xx + 0.5) * stride, (yy + 0.5) * stride), dim=-1).reshape(-1, 2))\n        stride_parts.append(torch.full((height * width,), float(stride), device=output.device, dtype=output.dtype))\n        level_parts.append(torch.full((height * width,), level, device=output.device, dtype=torch.long))\n    return {\n        "box_logits": torch.cat(box_parts, dim=1),\n        "class_logits": torch.cat(class_parts, dim=1),\n        "anchors": torch.cat(anchor_parts, dim=0),\n        "strides": torch.cat(stride_parts, dim=0),\n        "levels": torch.cat(level_parts, dim=0),\n    }\n\ndef decode_flat_boxes(flat, architecture):\n    reg_max = int(architecture["reg_max"])\n    logits = flat["box_logits"].reshape(flat["box_logits"].shape[0], -1, 4, reg_max)\n    probabilities = manual_softmax(logits, -1)\n    bins = torch.arange(reg_max, device=logits.device, dtype=logits.dtype)\n    distances = (probabilities * bins.reshape(1, 1, 1, -1)).sum(dim=-1)\n    distances = distances * flat["strides"].reshape(1, -1, 1)\n    anchors = flat["anchors"].reshape(1, -1, 2)\n    return torch.stack(\n        (\n            anchors[:, :, 0] - distances[:, :, 0],\n            anchors[:, :, 1] - distances[:, :, 1],\n            anchors[:, :, 0] + distances[:, :, 2],\n            anchors[:, :, 1] + distances[:, :, 3],\n        ),\n        dim=-1,\n    )\n\ndef pairwise_iou(boxes_a, boxes_b):\n    if boxes_a.numel() == 0 or boxes_b.numel() == 0:\n        return torch.zeros((boxes_a.shape[0], boxes_b.shape[0]), device=boxes_a.device)\n    left_top = torch.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])\n    right_bottom = torch.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])\n    intersection = (right_bottom - left_top).clamp_min(0.0).prod(dim=-1)\n    area_a = (boxes_a[:, 2:] - boxes_a[:, :2]).clamp_min(0.0).prod(dim=-1)\n    area_b = (boxes_b[:, 2:] - boxes_b[:, :2]).clamp_min(0.0).prod(dim=-1)\n    return intersection / (area_a[:, None] + area_b[None, :] - intersection).clamp_min(1e-9)\n\ndef manual_class_aware_nms(boxes, scores, classes, iou_threshold=0.6, max_detections=300):\n    kept = []\n    for class_id in torch.unique(classes).tolist():\n        indices = torch.nonzero(classes == class_id, as_tuple=False).reshape(-1)\n        indices = indices[torch.argsort(scores[indices], descending=True)]\n        while indices.numel() and len(kept) < max_detections:\n            current = int(indices[0].item())\n            kept.append(current)\n            if indices.numel() == 1:\n                break\n            remaining = indices[1:]\n            overlap = pairwise_iou(boxes[current : current + 1], boxes[remaining]).reshape(-1)\n            indices = remaining[overlap <= iou_threshold]\n    if not kept:\n        return torch.empty(0, dtype=torch.long, device=boxes.device)\n    keep = torch.tensor(kept, dtype=torch.long, device=boxes.device)\n    return keep[torch.argsort(scores[keep], descending=True)[:max_detections]]\n\ndef decode_predictions(outputs, architecture, confidence_threshold=0.25, iou_threshold=0.6, image_size=640, max_detections=300):\n    flat = flatten_detector_outputs(outputs, architecture)\n    boxes = decode_flat_boxes(flat, architecture).clamp(0.0, float(image_size))\n    class_probabilities = manual_sigmoid(flat["class_logits"])\n    scores, classes = class_probabilities.max(dim=-1)\n    results = []\n    for batch_index in range(boxes.shape[0]):\n        mask = scores[batch_index] >= confidence_threshold\n        selected_boxes = boxes[batch_index][mask]\n        selected_scores = scores[batch_index][mask]\n        selected_classes = classes[batch_index][mask]\n        if selected_scores.numel() > 3000:\n            order = torch.argsort(selected_scores, descending=True)[:3000]\n            selected_boxes, selected_scores, selected_classes = (\n                selected_boxes[order], selected_scores[order], selected_classes[order]\n            )\n        keep = manual_class_aware_nms(\n            selected_boxes, selected_scores, selected_classes, iou_threshold, max_detections\n        )\n        if keep.numel():\n            results.append(\n                torch.cat(\n                    (\n                        selected_boxes[keep],\n                        selected_scores[keep, None],\n                        selected_classes[keep, None].to(selected_boxes.dtype),\n                    ),\n                    dim=1,\n                )\n            )\n        else:\n            results.append(torch.empty((0, 6), device=boxes.device))\n    return results\n\ndef safe_load_checkpoint(path, device="cpu"):\n    try:\n        checkpoint = torch.load(path, map_location=device, weights_only=True)\n    except TypeError:\n        checkpoint = torch.load(path, map_location=device)\n    if not isinstance(checkpoint, dict) or checkpoint.get("format") != CUSTOM_CHECKPOINT_FORMAT:\n        raise ValueError(f"Not a {CUSTOM_CHECKPOINT_FORMAT} checkpoint: {path}")\n    required = {"class_names", "class_count", "architecture", "image_size", "model_state", "epoch"}\n    missing = sorted(required - set(checkpoint))\n    if missing:\n        raise ValueError(f"Checkpoint is missing required fields: {missing}")\n    return checkpoint\n\ndef model_from_checkpoint(path, device):\n    checkpoint = safe_load_checkpoint(path, device)\n    model = CustomDetector(checkpoint["architecture"]).to(device)\n    model.load_state_dict(checkpoint["model_state"])\n    model.eval()\n    return model, checkpoint\n\nIMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\nVIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".m4v"}\n\n\ndef preprocess_frame(frame, image_size):\n    original_height, original_width = frame.shape[:2]\n    scale = min(image_size / original_width, image_size / original_height)\n    resized_width = max(1, round(original_width * scale))\n    resized_height = max(1, round(original_height * scale))\n    resized = cv2.resize(frame, (resized_width, resized_height), interpolation=cv2.INTER_LINEAR)\n    pad_x = (image_size - resized_width) // 2\n    pad_y = (image_size - resized_height) // 2\n    canvas = np.full((image_size, image_size, 3), 114, dtype=np.uint8)\n    canvas[pad_y : pad_y + resized_height, pad_x : pad_x + resized_width] = resized\n    rgb = canvas[:, :, ::-1].copy()\n    tensor = torch.from_numpy(rgb).permute(2, 0, 1).to(torch.float32) / 255.0\n    transform = {\n        "scale": scale,\n        "pad_x": pad_x,\n        "pad_y": pad_y,\n        "original_width": original_width,\n        "original_height": original_height,\n    }\n    return tensor.unsqueeze(0), transform\n\n\ndef restore_detections(detections, transform):\n    restored = detections.detach().cpu().clone()\n    if restored.numel():\n        restored[:, [0, 2]] = (restored[:, [0, 2]] - transform["pad_x"]) / transform["scale"]\n        restored[:, [1, 3]] = (restored[:, [1, 3]] - transform["pad_y"]) / transform["scale"]\n        restored[:, [0, 2]] = restored[:, [0, 2]].clamp(0, transform["original_width"] - 1)\n        restored[:, [1, 3]] = restored[:, [1, 3]].clamp(0, transform["original_height"] - 1)\n    return restored\n\n\ndef infer_frame(model, architecture, frame, image_size, confidence, iou_threshold, device):\n    tensor, transform = preprocess_frame(frame, image_size)\n    with torch.no_grad():\n        outputs = model(tensor.to(device))\n        detections = decode_predictions(\n            outputs, architecture, confidence, iou_threshold, image_size, 300\n        )[0]\n    return restore_detections(detections, transform)\n\n\ndef annotate_frame(frame, detections, class_names):\n    colors = [(180, 180, 180), (239, 174, 0), (100, 100, 100), (68, 68, 239), (247, 85, 168), (11, 158, 245)]\n    result = frame.copy()\n    for row in detections:\n        x1, y1, x2, y2, score, class_value = row.tolist()\n        class_id = int(class_value)\n        color = colors[class_id % len(colors)]\n        label = f"{class_names[class_id]} {score:.3f}"\n        cv2.rectangle(result, (round(x1), round(y1)), (round(x2), round(y2)), color, 2)\n        cv2.putText(result, label, (round(x1), max(18, round(y1) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)\n    return result\n\n\ndef print_detections(source_name, detections, class_names):\n    print(f"{source_name}: {len(detections)} detections")\n    for row in detections:\n        x1, y1, x2, y2, score, class_value = row.tolist()\n        print(\n            f"  {class_names[int(class_value)]}: conf={score:.3f}, "\n            f"box_xyxy={[round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)]}"\n        )\n\n\ndef process_image(path, destination, model, architecture, class_names, image_size, confidence, iou_threshold, device):\n    frame = cv2.imread(str(path))\n    if frame is None:\n        raise ValueError(f"Could not read image: {path}")\n    detections = infer_frame(model, architecture, frame, image_size, confidence, iou_threshold, device)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if not cv2.imwrite(str(destination), annotate_frame(frame, detections, class_names)):\n        raise OSError(f"Could not write image: {destination}")\n    print_detections(path.name, detections, class_names)\n\n\ndef process_video(source, destination, model, architecture, class_names, image_size, confidence, iou_threshold, device):\n    capture = cv2.VideoCapture(source)\n    if not capture.isOpened():\n        raise ValueError(f"Could not open video/webcam source: {source}")\n    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))\n    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    fps = float(capture.get(cv2.CAP_PROP_FPS))\n    if not math.isfinite(fps) or fps <= 0:\n        fps = 25.0\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    writer = cv2.VideoWriter(str(destination), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))\n    frame_index = 0\n    try:\n        while True:\n            ok, frame = capture.read()\n            if not ok:\n                break\n            detections = infer_frame(model, architecture, frame, image_size, confidence, iou_threshold, device)\n            writer.write(annotate_frame(frame, detections, class_names))\n            if frame_index % 30 == 0:\n                print(f"frame {frame_index}: {len(detections)} detections")\n            frame_index += 1\n    finally:\n        capture.release()\n        writer.release()\n    print(f"Video frames processed: {frame_index}")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Run the custom school-uniform detector on images, folders, videos, or a webcam.")\n    parser.add_argument("--source", required=True, help="Image, folder, video, or webcam index such as 0.")\n    parser.add_argument("--weights", default="best.pt", help="Custom checkpoint path.")\n    parser.add_argument("--conf", type=float, default=0.25, help="Confidence threshold.")\n    parser.add_argument("--iou", type=float, default=0.60, help="Suppression overlap threshold.")\n    parser.add_argument("--imgsz", type=int, default=None, help="Input size; default comes from the checkpoint.")\n    parser.add_argument("--output", default="runs_uniform_predict", help="Parent output folder.")\n    parser.add_argument("--device", default="auto", help="auto, cpu, or cuda.")\n    args = parser.parse_args()\n\n    package_dir = Path(__file__).resolve().parent\n    weights_path = Path(args.weights).expanduser()\n    if not weights_path.is_absolute():\n        weights_path = package_dir / weights_path\n    if not weights_path.is_file():\n        raise FileNotFoundError(f"Weights not found: {weights_path}")\n    device = "cuda" if args.device == "auto" and torch.cuda.is_available() else ("cpu" if args.device == "auto" else args.device)\n    model, checkpoint = model_from_checkpoint(weights_path, device)\n    architecture = checkpoint["architecture"]\n    class_names = list(checkpoint["class_names"])\n    image_size = int(args.imgsz or checkpoint["image_size"])\n    if image_size % 32:\n        raise ValueError("--imgsz must be divisible by 32.")\n    output_dir = Path(args.output).expanduser() / "predict"\n    source_text = args.source.strip()\n\n    if source_text.isdigit():\n        destination = output_dir / f"webcam_{source_text}.mp4"\n        process_video(int(source_text), destination, model, architecture, class_names, image_size, args.conf, args.iou, device)\n    else:\n        source_path = Path(source_text).expanduser()\n        if not source_path.exists():\n            raise FileNotFoundError(f"Source path not found: {source_path}")\n        if source_path.is_dir():\n            image_paths = sorted(path for path in source_path.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)\n            for path in image_paths:\n                process_image(path, output_dir / path.name, model, architecture, class_names, image_size, args.conf, args.iou, device)\n        elif source_path.suffix.lower() in IMAGE_EXTENSIONS:\n            process_image(source_path, output_dir / source_path.name, model, architecture, class_names, image_size, args.conf, args.iou, device)\n        elif source_path.suffix.lower() in VIDEO_EXTENSIONS:\n            process_video(str(source_path), output_dir / f"{source_path.stem}.mp4", model, architecture, class_names, image_size, args.conf, args.iou, device)\n        else:\n            raise ValueError(f"Unsupported source: {source_path}")\n    print(f"Annotated outputs saved to: {output_dir}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
PACKAGE_ROOT = Path(globals().get("PACKAGE_ROOT_OVERRIDE", "/content/uniform_windows_package"))
ZIP_PATH = Path(globals().get("ZIP_PATH_OVERRIDE", ZIP_PATH))
PACKAGE_REPORT_PATH = OUTPUT_ROOT / "windows_package_build_report.json"
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)

for checkpoint_path in (Path(BEST_PT), Path(LAST_PT)):
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Required custom checkpoint missing: {checkpoint_path}")
    safe_load_checkpoint(checkpoint_path, "cpu")
    shutil.copy2(checkpoint_path, PACKAGE_ROOT / checkpoint_path.name)

package_yaml = {
    "path": ".",
    "train": "images/train",
    "val": "images/val",
    "nc": NUM_CLASSES,
    "names": {index: name for index, name in enumerate(CLASS_NAMES)},
}
(PACKAGE_ROOT / "data.yaml").write_text(yaml.safe_dump(package_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")
(PACKAGE_ROOT / "classes.txt").write_text("\n".join(CLASS_NAMES) + "\n", encoding="utf-8")
write_json(PACKAGE_ROOT / "class_names.json", {index: name for index, name in enumerate(CLASS_NAMES)})
(PACKAGE_ROOT / "infer_windows.py").write_text(INFER_WINDOWS_SOURCE, encoding="utf-8")
(PACKAGE_ROOT / "requirements_windows.txt").write_text("torch\nnumpy\nopencv-python\n", encoding="utf-8")
(PACKAGE_ROOT / "run_example_windows.bat").write_text(
    "@echo off\n"
    "REM Image inference\npython infer_windows.py --source demo_image.jpg --weights best.pt --conf 0.25\n\n"
    "REM Folder inference\npython infer_windows.py --source demo_images --weights best.pt --conf 0.25\n\n"
    "REM Video inference\npython infer_windows.py --source demo_video.mp4 --weights best.pt --conf 0.25\n\n"
    "REM Webcam inference\npython infer_windows.py --source 0 --weights best.pt --conf 0.25\n\npause\n",
    encoding="utf-8",
)
class_mapping_text = "\n".join(f"{index}: {name}" for index, name in enumerate(CLASS_NAMES))
readme = f"""# School Uniform Custom Detector - Windows 11 64-bit

Use 64-bit Python 3.10 or 3.11. This package contains a custom anchor-free detector trained from random He initialization. Its convolution, normalization, activations, C2f/SPPF/PAN-FPN network, decode, and class-aware suppression are implemented directly with tensor arithmetic. No external detector implementation or prior detector checkpoint is an initialization source.

## Install
```bat
python -m venv .venv
.venv\Scripts\activate
python -m pip install --upgrade pip
pip install -r requirements_windows.txt
```

## Run
```bat
python infer_windows.py --source demo_image.jpg --weights best.pt --conf 0.25
python infer_windows.py --source demo_images --weights best.pt --conf 0.25
python infer_windows.py --source demo_video.mp4 --weights best.pt --conf 0.25
python infer_windows.py --source 0 --weights best.pt --conf 0.25
```

Annotated output is written under `runs_uniform_predict/predict` by default.

## Class Mapping
{class_mapping_text}

`best.pt` is recommended for inference. `last.pt` includes manual optimizer and training state for continuation/audit. `data.yaml` contains train/validation metadata only. Raw images and source labels are intentionally excluded.
"""
(PACKAGE_ROOT / "README_WINDOWS_11.md").write_text(readme, encoding="utf-8")

checkpoint_status = {
    "best_pt": str(BEST_PT), "best_pt_included": True,
    "last_pt": str(LAST_PT), "last_pt_included": True,
    "note": "Both files use the custom checkpoint dictionary format.",
}
write_json(PACKAGE_ROOT / "checkpoint_status.json", checkpoint_status)
write_json(
    PACKAGE_ROOT / "model_provenance.json",
    {
        "deployment_model": "best.pt",
        "model_format": CUSTOM_CHECKPOINT_FORMAT,
        "framework": "custom low-level tensor runtime",
        "training_method": "random initialization and custom training mathematics",
        "initialization": "random He initialization",
        "external_model_source": False,
        "prior_detector_weights_used": False,
        "nc": NUM_CLASSES,
        "class_names": {index: name for index, name in enumerate(CLASS_NAMES)},
        "architecture": MODEL_CONFIG,
        "checkpoint_status": checkpoint_status,
    },
)

artifact_names = [
    "results.csv", "validation_metrics.json", "validation_summary.csv", "validation_artifact_report.json",
    "confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png",
    "training_curves.png", "class_distribution.png", "class_distribution.csv", "split_summary.csv", "split_manifest.csv",
    "dataset_validation_report.json", "dataset_validation_report.csv", "dataset_overview.json", "training_configuration.json",
    "training_run_metadata.json", "validation_run_metadata.json", "training_args.yaml",
]
for artifact_name in artifact_names:
    source_path = OUTPUT_ROOT / artifact_name
    if source_path.is_file():
        shutil.copy2(source_path, PACKAGE_ROOT / artifact_name)
for directory_name in ("sample_predictions", "bbox_preview_samples"):
    source_dir = OUTPUT_ROOT / directory_name
    if source_dir.is_dir():
        shutil.copytree(source_dir, PACKAGE_ROOT / directory_name, dirs_exist_ok=True)
package_log_dir = PACKAGE_ROOT / "framework_logs"
for log_value in (install_log, TRAINING_LOG_PATH, VALIDATION_LOG_PATH, SAMPLE_INFERENCE_LOG_PATH):
    log_path = Path(log_value)
    if log_path.is_file():
        package_log_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(log_path, package_log_dir / log_path.name)

essential_entries = [
    "best.pt", "data.yaml", "classes.txt", "class_names.json", "infer_windows.py",
    "requirements_windows.txt", "run_example_windows.bat", "README_WINDOWS_11.md",
    "model_provenance.json", "checkpoint_status.json", "package_manifest.json", "checksums.json",
]
manifest_files = sorted(
    str(path.relative_to(PACKAGE_ROOT)).replace("\\", "/")
    for path in PACKAGE_ROOT.rglob("*") if path.is_file()
)
manifest_files.extend(["package_manifest.json", "checksums.json", "windows_package_build_report.json"])
package_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "target_system": "Windows 11 64-bit",
    "model_format": CUSTOM_CHECKPOINT_FORMAT,
    "training_method": "random initialization with custom tensor mathematics",
    "external_model_source": False,
    "prior_detector_weights_used": False,
    "model_weights": ["best.pt", "last.pt"],
    "nc": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "essential_entries": essential_entries,
    "files": sorted(set(manifest_files)),
    "warnings": [],
}
write_json(PACKAGE_ROOT / "package_manifest.json", package_manifest)
package_report = {
    "package_created": True,
    "zip_path": str(ZIP_PATH),
    "package_root": str(PACKAGE_ROOT),
    "best_pt": str(BEST_PT),
    "last_pt": str(LAST_PT),
    "checkpoint_status": checkpoint_status,
    "warnings": [],
    "errors": [],
}
write_json(PACKAGE_ROOT / "windows_package_build_report.json", package_report)
write_json(PACKAGE_REPORT_PATH, package_report)
checksums = {
    str(path.relative_to(PACKAGE_ROOT)).replace("\\", "/"): sha256_file(path)
    for path in sorted(PACKAGE_ROOT.rglob("*"))
    if path.is_file() and path.name != "checksums.json"
}
write_json(PACKAGE_ROOT / "checksums.json", checksums)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PACKAGE_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, str(path.relative_to(PACKAGE_ROOT)).replace("\\", "/"))
with zipfile.ZipFile(ZIP_PATH) as archive:
    names = set(archive.namelist())
    missing = sorted(set(essential_entries) - names)
    if missing:
        raise RuntimeError(f"ZIP is missing essential entries: {missing}")
    packaged_yaml = yaml.safe_load(archive.read("data.yaml").decode("utf-8")) or {}
    if "test" in packaged_yaml or "test.txt" in names or "test.csv" in names:
        raise RuntimeError("A test split artifact was found in the package.")
    forbidden_raw = [name for name in names if name.startswith("dataset_images/") or name.startswith("labels/all/")]
    if forbidden_raw:
        raise RuntimeError(f"Raw dataset content leaked into the package: {forbidden_raw[:5]}")
    recorded = json.loads(archive.read("checksums.json"))
    for name, expected in recorded.items():
        actual = hashlib.sha256(archive.read(name)).hexdigest()
        if actual != expected:
            raise RuntimeError(f"Checksum mismatch for {name}")

print("Windows deployment package created.")
print(f"Package path: {ZIP_PATH}")
print(f"Package size MB: {ZIP_PATH.stat().st_size / (1024 ** 2):.2f}")
print(f"BEST_PT included: {BEST_PT}")
print(f"LAST_PT included: {LAST_PT}")
print(f"Build report: {PACKAGE_REPORT_PATH}")
try:
    if colab_files is not None:
        colab_files.download(str(ZIP_PATH))
except Exception as exc:
    print(f"WARNING: ZIP was created but automatic browser download did not start: {exc}")


Windows deployment package created.
Package path: /content/drive/MyDrive/DATN2/yolov8_uniform_windows_package.zip
Package size MB: 53.52
BEST_PT included: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/best.pt
LAST_PT included: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/last.pt
Package warnings: 1
Build report: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/windows_package_build_report.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>